# Notebook 17 — Runner Characteristics and Equipment

## Bounded question

> What do the runner-level `age`, `sex` and `hg` fields represent in the source, how consistently are they populated across jurisdictions and racing types, and which values can be normalised or derived safely without inventing official eligibility, identity or equipment facts?

## Initial governed scope

This notebook investigates three runner-level source fields:

- `age`
- `sex`
- `hg`

The source-field governance register assigns all three to the `runner_identity_and_characteristics` family and requires their raw values to be preserved.

The investigation begins with source profiling only. At this stage, no assumption is made that:

- `age` follows one universal international racing convention;
- runner `sex` codes have globally stable meanings;
- `hg` means headgear or equipment;
- blank values mean “none” rather than unknown or not supplied;
- runner characteristics prove official race eligibility.

Profiling evidence will be kept separate from interpretation. Any normalisation rule must preserve the original source value, expose unresolved cases explicitly and respect jurisdiction or period differences where the evidence requires them.

## Stage 1 — Source lineage and governed population

This stage establishes the physical source, read-only controls and expected population before profiling `age`, `sex` or `hg`.

The immutable source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`

The established source population is:

- 1,851,285 runner rows;
- 189,043 provisional races;
- 37 source columns;
- provisional race identity: `date + course + off`.

The first code cell will open the SQLite database in read-only mode, confirm the source table and columns, and reconcile the governed runner and provisional-race counts. No field interpretation will be attempted yet.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)

source_columns = pd.read_sql_query(
    f"PRAGMA table_info({SOURCE_TABLE})",
    connection,
)

population = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            CAST(date AS TEXT) || '|' ||
            CAST(course AS TEXT) || '|' ||
            CAST(off AS TEXT)
        ) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

observed_runner_rows = int(population.loc[0, "runner_rows"])
observed_provisional_races = int(population.loc[0, "provisional_races"])
observed_source_columns = len(source_columns)

assert observed_runner_rows == EXPECTED_RUNNER_ROWS
assert observed_provisional_races == EXPECTED_PROVISIONAL_RACES
assert observed_source_columns == EXPECTED_SOURCE_COLUMNS

print("Governed source population confirmed")
display(
    pd.DataFrame(
        {
            "measure": [
                "source database",
                "source table",
                "data-row predicate",
                "runner rows",
                "provisional races",
                "source columns",
            ],
            "value": [
                str(SOURCE_DB_PATH.relative_to(PROJECT_ROOT)),
                SOURCE_TABLE,
                DATA_ROW_PREDICATE,
                observed_runner_rows,
                observed_provisional_races,
                observed_source_columns,
            ],
        }
    )
)

Governed source population confirmed


,measure,value
0,source database,data/raw/form_2015-present/form_2015-present/r...
1,source table,data
2,data-row predicate,rowid <> 1
3,runner rows,1851285
4,provisional races,189043
5,source columns,37


## Stage 2 — Confirm governed field ownership and declared source types

Before examining values, this stage checks the existing source-field governance rows for `age`, `sex` and `hg`.

The purpose is to confirm:

- runner-level grain;
- declared SQLite type;
- field-family ownership;
- raw-value preservation requirements;
- current blank, dash and zero policies;
- semantic status before Notebook 17 interpretation.

These governance rows are starting constraints, not final semantic conclusions. Notebook 17 may refine their interpretation, but it must not bypass raw preservation or silently replace unresolved policies.

In [2]:
# Locate the existing governed source-field inventory created during Notebook 02.
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "source_field_governance.csv"
)

# Fail immediately if the governed reference is missing rather than
# reconstructing a private field inventory inside this notebook.
if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        "Source-field governance reference not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

# Load the permanent field-governance reference.
source_field_governance = pd.read_csv(SOURCE_FIELD_GOVERNANCE_PATH)

# Confirm that the reference still contains the policy columns required
# to interpret its rows safely.
required_governance_columns = {
    "source_field",
    "declared_type",
    "grain",
    "field_family",
    "raw_preservation",
    "blank_policy",
    "dash_policy",
    "zero_policy",
    "governed_by",
    "status",
}

missing_governance_columns = (
    required_governance_columns
    - set(source_field_governance.columns)
)

if missing_governance_columns:
    raise ValueError(
        "Source-field governance reference is missing required columns: "
        f"{sorted(missing_governance_columns)}"
    )

# Preserve the investigation order used by the bounded question rather
# than relying on the source CSV's row order.
runner_characteristic_fields = ["age", "sex", "hg"]

# Extract only the existing governance rows relevant to Notebook 17.
runner_characteristic_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(
            runner_characteristic_fields
        ),
        [
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .set_index("source_field")
    .loc[runner_characteristic_fields]
    .reset_index()
)

# Reconcile the extracted rows with the established governance contract.
assert len(runner_characteristic_governance) == 3
assert set(runner_characteristic_governance["grain"]) == {"runner"}
assert set(
    runner_characteristic_governance["field_family"]
) == {"runner_identity_and_characteristics"}
assert set(
    runner_characteristic_governance["raw_preservation"]
) == {"required"}

# Display the starting governance position without treating it as a final
# semantic conclusion for age, sex or hg.
display(runner_characteristic_governance)

,source_field,declared_type,grain,field_family,raw_preservation,blank_policy,dash_policy,zero_policy,governed_by,status
0,age,INTEGER,runner,runner_identity_and_characteristics,required,unresolved_missing,not_expected,possible_sentinel,Notebook 10,pending_semantics
1,sex,TEXT,runner,runner_identity_and_characteristics,required,unresolved_missing,not_expected,contextual_value,Notebook 10,pending_semantics
2,hg,TEXT,runner,runner_identity_and_characteristics,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics


## Stage 3 — Profile raw storage and availability

This stage examines the source values before assigning any domain meaning.

For each of `age`, `sex` and `hg`, it will establish:

- SQLite storage classes;
- SQL null frequency;
- blank-text frequency;
- populated-row frequency;
- number of distinct populated raw values.

The fields will be profiled independently because their missing-value policies differ. In particular, a blank `hg` value must not yet be interpreted as “no headgear” or “no equipment.”

In [3]:
# Profile storage and availability independently for each Notebook 17 field.
#
# SQL NULL and blank text are counted separately because the governed
# source-field reference does not permit one universal missing-value rule.
field_storage_profiles = []

for field_name in runner_characteristic_fields:
    field_profile = pd.read_sql_query(
        f"""
        SELECT
            '{field_name}' AS source_field,
            COUNT(*) AS runner_rows,
            SUM(CASE WHEN {field_name} IS NULL THEN 1 ELSE 0 END) AS null_rows,
            SUM(
                CASE
                    WHEN {field_name} IS NOT NULL
                     AND TRIM(CAST({field_name} AS TEXT)) = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_text_rows,
            SUM(
                CASE
                    WHEN {field_name} IS NOT NULL
                     AND TRIM(CAST({field_name} AS TEXT)) <> ''
                    THEN 1
                    ELSE 0
                END
            ) AS populated_rows,
            COUNT(
                DISTINCT CASE
                    WHEN {field_name} IS NOT NULL
                     AND TRIM(CAST({field_name} AS TEXT)) <> ''
                    THEN {field_name}
                END
            ) AS distinct_populated_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    )

    field_storage_profiles.append(field_profile)

# Combine the three independently calculated field summaries.
runner_characteristic_availability = pd.concat(
    field_storage_profiles,
    ignore_index=True,
)

# Confirm that every field profile reconciles to the governed runner population.
assert (
    runner_characteristic_availability["runner_rows"]
    == EXPECTED_RUNNER_ROWS
).all()

assert (
    runner_characteristic_availability[
        ["null_rows", "blank_text_rows", "populated_rows"]
    ].sum(axis=1)
    == EXPECTED_RUNNER_ROWS
).all()

display(runner_characteristic_availability)

,source_field,runner_rows,null_rows,blank_text_rows,populated_rows,distinct_populated_values
0,age,1851285,0,0,1851285,19
1,sex,1851285,0,0,1851285,8
2,hg,1851285,0,1122490,728795,60


## Stage 4 — Inspect SQLite storage classes and complete raw vocabularies

The availability profile shows that `age` and `sex` are populated on every governed runner row, while `hg` uses blank text extensively.

The next stage examines:

- the SQLite storage classes actually used by each field;
- the complete observed vocabulary for `age` and `sex`;
- the frequency distribution of populated `hg` values;
- whether declared source types match physical storage;
- whether apparently simple fields contain mixed or malformed representations.

No raw value will yet be mapped to a racing meaning. The purpose is to establish the exact source vocabulary before interpreting codes or constructing parsers.

In [4]:
# Count the physical SQLite storage classes used by each Notebook 17 field.
#
# SQLite declared types do not guarantee that every stored value uses the
# corresponding physical storage class, so the source must be checked directly.
storage_class_profiles = []

for field_name in runner_characteristic_fields:
    storage_profile = pd.read_sql_query(
        f"""
        SELECT
            '{field_name}' AS source_field,
            TYPEOF({field_name}) AS storage_class,
            COUNT(*) AS runner_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY TYPEOF({field_name})
        ORDER BY runner_rows DESC, storage_class
        """,
        connection,
    )

    storage_class_profiles.append(storage_profile)

runner_characteristic_storage_classes = pd.concat(
    storage_class_profiles,
    ignore_index=True,
)

# Confirm that the storage-class counts reconcile independently for each field.
storage_class_totals = (
    runner_characteristic_storage_classes
    .groupby("source_field", as_index=False)["runner_rows"]
    .sum()
)

assert set(storage_class_totals["source_field"]) == set(
    runner_characteristic_fields
)
assert (
    storage_class_totals["runner_rows"] == EXPECTED_RUNNER_ROWS
).all()

print("SQLite storage classes")
display(runner_characteristic_storage_classes)

# Display the complete raw vocabulary for age and sex because both fields
# contain only a small number of distinct populated values.
age_vocabulary = pd.read_sql_query(
    f"""
    SELECT
        age AS raw_age,
        TYPEOF(age) AS storage_class,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY age, TYPEOF(age)
    ORDER BY
        CASE WHEN TYPEOF(age) IN ('integer', 'real') THEN age END,
        CAST(age AS TEXT)
    """,
    connection,
)

sex_vocabulary = pd.read_sql_query(
    f"""
    SELECT
        sex AS raw_sex,
        TYPEOF(sex) AS storage_class,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY sex, TYPEOF(sex)
    ORDER BY runner_rows DESC, CAST(sex AS TEXT)
    """,
    connection,
)

# Show the populated hg vocabulary in descending frequency order.
#
# Blank text remains excluded from this table because its frequency has
# already been established separately and its meaning remains unresolved.
hg_vocabulary = pd.read_sql_query(
    f"""
    SELECT
        hg AS raw_hg,
        TYPEOF(hg) AS storage_class,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(hg AS TEXT)) <> ''
    GROUP BY hg, TYPEOF(hg)
    ORDER BY runner_rows DESC, CAST(hg AS TEXT)
    """,
    connection,
)

assert len(age_vocabulary) == 19
assert len(sex_vocabulary) == 8
assert len(hg_vocabulary) == 60

print("Complete age vocabulary")
display(age_vocabulary)

print("Complete sex vocabulary")
display(sex_vocabulary)

print("Complete populated hg vocabulary")
display(hg_vocabulary)

SQLite storage classes


,source_field,storage_class,runner_rows
0,age,integer,1851285
1,sex,text,1851285
2,hg,text,1851285


Complete age vocabulary


,raw_age,storage_class,runner_rows
0,1,integer,5
1,2,integer,180383
2,3,integer,400774
3,4,integer,342278
4,5,integer,299403
5,6,integer,231753
6,7,integer,161945
7,8,integer,105400
8,9,integer,64630
9,10,integer,35998


Complete sex vocabulary


,raw_sex,storage_class,runner_rows
0,G,text,1078420
1,F,text,371961
2,M,text,190797
3,C,text,178499
4,H,text,30728
5,R,text,878
6,B,text,1
7,BB,text,1


Complete populated hg vocabulary


,raw_hg,storage_class,runner_rows
0,t,text,178727
1,p,text,173713
2,b,text,128859
3,h,text,62258
4,tp,text,52073
5,v,text,40585
6,tb,text,40299
7,ht,text,22190
8,tv,text,9729
9,e/s,text,2943


## Stage 5 — Establish the documented code system before testing exceptions

The raw vocabularies are consistent with a structured racecard code system rather than unrestricted text.

Published Racing Post racecard guidance documents the common runner-sex codes:

- `C` — colt;
- `F` — filly;
- `G` — gelding;
- `H` — horse;
- `M` — mare;
- `R` — rig.

It also documents the principal headgear codes:

- `b` — blinkers;
- `p` — cheekpieces;
- `t` — tongue-tie;
- `v` — visor;
- `h` — hood;
- `e` — eye hood;
- `e/c` — eyecover;
- `e/s` — eyeshield.

The same guidance states that a following `1` or `2` can indicate first- or second-time use for applicable headgear. :contentReference[oaicite:0]{index=0}

This is external interpretive evidence, not proof that every value in this database follows the guidance universally. The next stage will therefore test the complete source vocabulary against the documented codes while preserving:

- original case;
- original token order;
- combined codes;
- numeric suffixes;
- undocumented or jurisdiction-specific values;
- blank values as unresolved until their source meaning is demonstrated.

The rare `B` and `BB` runner-sex values will not be labelled erroneous merely because they are absent from this initial reference. They require source-context and, if necessary, separately governed verification.

## Stage 6 — Preserve external code-reference evidence

The interpretation of the `sex` and `hg` vocabularies now depends partly on published racecard guidance rather than source-internal profiling alone.

The project procedure therefore requires the external evidence to be captured while it is open, with permanent verification identifiers and explicit downstream authority.

Two bounded claims will be registered:

- `NB17-SEX-0001` — published Racing Post guidance defines the common runner-sex abbreviations `C`, `F`, `G`, `H`, `M` and `R`;
- `NB17-HG-0001` — published racecard guidance defines the principal headgear codes and identifies a trailing `1` as first-time use under that code.

These verification records support construction of a governed code reference. They do not establish that every observed source value is covered, universally applicable across all jurisdictions, or safe to normalise without further source-wide testing.

The immutable source values will remain unchanged.

In [5]:
# Reload the already-persisted Notebook 17 external code evidence.
#
# These records were written successfully before the kernel restart.
# This cell does not append, overwrite or reconstruct them.
MANUAL_VERIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "manual_verifications.csv"
)

manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    dtype=str,
    keep_default_na=False,
)

external_code_verification_ids = {
    "NB17-SEX-0001",
    "NB17-HG-0001",
}

reloaded_external_code_verifications = (
    manual_verifications.loc[
        manual_verifications["verification_id"].isin(
            external_code_verification_ids
        )
    ]
    .sort_values("verification_id")
    .reset_index(drop=True)
)

# Confirm that both permanent records exist exactly once.
assert len(reloaded_external_code_verifications) == 2
assert set(
    reloaded_external_code_verifications["verification_id"]
) == external_code_verification_ids
assert set(
    reloaded_external_code_verifications["verification_status"]
) == {"confirmed"}
assert set(
    reloaded_external_code_verifications["database_action"]
) == {"reference_enrichment"}

print(
    "Notebook 17 external code evidence "
    "already persisted and successfully reloaded"
)

display(
    reloaded_external_code_verifications[
        [
            "verification_id",
            "source_field",
            "raw_source_value",
            "verified_value",
            "verification_status",
            "confidence",
            "database_action",
        ]
    ]
)

Notebook 17 external code evidence already persisted and successfully reloaded


,verification_id,source_field,raw_source_value,verified_value,verification_status,confidence,database_action
0,NB17-HG-0001,hg,h|b|p|t|v|e|ht|e/c|e/s|b1|b2,h=hood; b=blinkers; p=cheekpieces; t=tongue-ti...,confirmed,high,reference_enrichment
1,NB17-SEX-0001,sex,C|F|G|H|M|R,C=colt; F=filly; G=gelding; H=horse; M=mare; R...,confirmed,high,reference_enrichment


## Stage 7 — Test the runner-sex vocabulary against the verified reference

The source contains eight runner-sex codes:

- six codes covered by `NB17-SEX-0001`: `C`, `F`, `G`, `H`, `M`, `R`;
- two codes not covered by that reference: `B`, `BB`.

This stage will separate:

- values supported by the verified common-code mapping;
- values that remain unresolved;
- the row and race coverage of each state;
- the jurisdiction, period and runner context of the unresolved residue.

Absence from the initial reference does not make `B` or `BB` erroneous. They will remain raw, explicit and unresolved until source context or additional evidence establishes their meaning.

In [6]:
# Define only the six runner-sex mappings supported by NB17-SEX-0001.
#
# B and BB are deliberately excluded rather than guessed from their spelling.
verified_sex_code_map = {
    "C": "colt",
    "F": "filly",
    "G": "gelding",
    "H": "horse",
    "M": "mare",
    "R": "rig",
}

# Classify the complete observed sex vocabulary against the verified mapping.
sex_vocabulary_governance = sex_vocabulary.copy()

sex_vocabulary_governance["normalised_sex"] = (
    sex_vocabulary_governance["raw_sex"].map(verified_sex_code_map)
)

sex_vocabulary_governance["interpretation_status"] = (
    sex_vocabulary_governance["normalised_sex"]
    .notna()
    .map(
        {
            True: "verified_common_code",
            False: "unresolved_source_code",
        }
    )
)

sex_vocabulary_governance["verification_id"] = (
    sex_vocabulary_governance["interpretation_status"]
    .map(
        {
            "verified_common_code": "NB17-SEX-0001",
            "unresolved_source_code": "",
        }
    )
)

# Confirm that all eight observed values are partitioned exactly once.
assert len(sex_vocabulary_governance) == 8
assert set(sex_vocabulary_governance["raw_sex"]) == {
    "C",
    "F",
    "G",
    "H",
    "M",
    "R",
    "B",
    "BB",
}
assert (
    sex_vocabulary_governance["runner_rows"].sum()
    == EXPECTED_RUNNER_ROWS
)

print("Runner-sex vocabulary governance")
display(
    sex_vocabulary_governance[
        [
            "raw_sex",
            "runner_rows",
            "normalised_sex",
            "interpretation_status",
            "verification_id",
        ]
    ]
)

# Summarise how much of the governed runner population is covered by the
# externally verified common-code mapping.
sex_coverage_summary = (
    sex_vocabulary_governance
    .groupby("interpretation_status", as_index=False)
    .agg(
        distinct_raw_values=("raw_sex", "nunique"),
        runner_rows=("runner_rows", "sum"),
    )
)

sex_coverage_summary["runner_row_percentage"] = (
    sex_coverage_summary["runner_rows"]
    / EXPECTED_RUNNER_ROWS
    * 100
)

assert sex_coverage_summary["runner_rows"].sum() == EXPECTED_RUNNER_ROWS

print("Runner-sex interpretation coverage")
display(sex_coverage_summary)

# Inspect every row carrying a sex code not covered by NB17-SEX-0001.
#
# Jurisdiction is not inferred directly from course text here. The source
# context is preserved for inspection before any further external research.
unresolved_sex_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        age_band,
        sex_rest,
        horse,
        age,
        sex,
        hg,
        num,
        pos,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND sex IN ('B', 'BB')
    ORDER BY date, course, off, horse, source_rowid
    """,
    connection,
)

assert len(unresolved_sex_rows) == 2
assert set(unresolved_sex_rows["sex"]) == {"B", "BB"}

print("Unresolved runner-sex rows")
display(unresolved_sex_rows)

Runner-sex vocabulary governance


,raw_sex,runner_rows,normalised_sex,interpretation_status,verification_id
0,G,1078420,gelding,verified_common_code,NB17-SEX-0001
1,F,371961,filly,verified_common_code,NB17-SEX-0001
2,M,190797,mare,verified_common_code,NB17-SEX-0001
3,C,178499,colt,verified_common_code,NB17-SEX-0001
4,H,30728,horse,verified_common_code,NB17-SEX-0001
5,R,878,rig,verified_common_code,NB17-SEX-0001
6,B,1,NaN,unresolved_source_code,
7,BB,1,NaN,unresolved_source_code,


Runner-sex interpretation coverage


,interpretation_status,distinct_raw_values,runner_rows,runner_row_percentage
0,unresolved_source_code,2,2,0.000108
1,verified_common_code,6,1851283,99.999892


Unresolved runner-sex rows


,source_rowid,date,course,off,race_id,race_name,type,age_band,sex_rest,horse,age,sex,hg,num,pos,comment
0,448648,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Par Coeur (GER),3,BB,,10,5,
1,814999,2019-11-29,Gulfstream Park (USA),8:30,746195,Starter Optional Claiming (Claimer) (2yo Filli...,Flat,2yo,F,La Venezolana (VEN),2,B,b,2,7,


## Stage 8 — Test unresolved sex codes against repeated horse histories

The two unresolved sex codes occur in different jurisdictions:

- `BB` — Par Coeur (GER), Cologne, 15 October 2017;
- `B` — La Venezolana (VEN), Gulfstream Park, 29 November 2019.

Before consulting another external reference, this stage tests whether the same source horse labels appear elsewhere with one of the six verified runner-sex codes.

Repeated-horse evidence can reveal whether:

- the unresolved code is stable for that horse;
- another source record supplies a verified sex code;
- the value appears to be a one-row source inconsistency;
- the horse label itself is ambiguous or reused.

A different code on another row will not automatically authorise correction. Horse labels are source-presented identities rather than proven permanent entity keys, and any conclusion must retain the exact race and row lineage.

In [7]:
# Retrieve every governed source appearance of the two horses carrying the
# unresolved sex codes B and BB.
#
# Horse text is used only as a source-label lookup here. It is not treated as
# a proven permanent horse entity key.
unresolved_sex_horses = (
    unresolved_sex_rows["horse"]
    .drop_duplicates()
    .tolist()
)

assert unresolved_sex_horses == [
    "Par Coeur (GER)",
    "La Venezolana (VEN)",
]

placeholders = ", ".join("?" for _ in unresolved_sex_horses)

unresolved_horse_histories = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        horse,
        age,
        sex,
        hg,
        num,
        pos,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND horse IN ({placeholders})
    ORDER BY horse, date, course, off, source_rowid
    """,
    connection,
    params=unresolved_sex_horses,
)

# Confirm that both source horse labels were recovered.
assert set(unresolved_horse_histories["horse"]) == set(
    unresolved_sex_horses
)

print("Complete source histories for unresolved-sex horse labels")
display(unresolved_horse_histories)

# Summarise the sex vocabulary recorded across each horse label's history.
unresolved_horse_sex_summary = (
    unresolved_horse_histories
    .groupby(["horse", "sex"], as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_courses=("course", "nunique"),
    )
    .sort_values(
        ["horse", "runner_rows", "sex"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

print("Sex-code history by unresolved horse label")
display(unresolved_horse_sex_summary)

Complete source histories for unresolved-sex horse labels


,source_rowid,date,course,off,race_id,race_name,type,horse,age,sex,hg,num,pos,comment
0,814999,2019-11-29,Gulfstream Park (USA),8:30,746195,Starter Optional Claiming (Claimer) (2yo Filli...,Flat,La Venezolana (VEN),2,B,b,2,7,
1,448648,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,Par Coeur (GER),3,BB,,10,5,


Sex-code history by unresolved horse label


,horse,sex,runner_rows,first_date,last_date,distinct_courses
0,La Venezolana (VEN),B,1,2019-11-29,2019-11-29,1
1,Par Coeur (GER),BB,1,2017-10-15,2017-10-15,1


## Stage 9 — Inspect unresolved codes within their complete race context

Neither unresolved horse label appears elsewhere in the source, and the initial external search did not establish the meanings of `B` or `BB`.

The next source-internal check therefore examines every runner in the two affected races.

This can show whether:

- the unresolved values are isolated within otherwise standard sex-code vocabularies;
- the race-level `sex_rest` value supplies relevant context;
- the race name describes a sex restriction;
- neighbouring runner rows reveal a jurisdiction-specific coding pattern;
- the values look more like isolated source substitutions than stable additional categories.

Race-level restrictions will be treated as contextual evidence only. They cannot, by themselves, define the runner-level code or prove an official eligibility fact.

In [8]:
# Recover the provisional race identities containing the unresolved sex codes.
unresolved_sex_race_keys = (
    unresolved_sex_rows[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert len(unresolved_sex_race_keys) == 2

# Load every runner row from each affected provisional race.
#
# The query is deliberately bounded to the two exact date-course-off
# identities rather than searching broadly for similar race names.
unresolved_race_context_frames = []

for race_key in unresolved_sex_race_keys.itertuples(index=False):
    race_context = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            type,
            age_band,
            sex_rest,
            horse,
            age,
            sex,
            hg,
            num,
            pos,
            comment
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND date = ?
          AND course = ?
          AND off = ?
        ORDER BY
            CASE
                WHEN TYPEOF(pos) = 'integer' THEN pos
                ELSE 999
            END,
            num,
            horse,
            source_rowid
        """,
        connection,
        params=[
            race_key.date,
            race_key.course,
            race_key.off,
        ],
    )

    unresolved_race_context_frames.append(race_context)

unresolved_sex_race_context = pd.concat(
    unresolved_race_context_frames,
    ignore_index=True,
)

# Confirm that both unresolved rows are present in the recovered race contexts.
recovered_unresolved_rows = unresolved_sex_race_context.loc[
    unresolved_sex_race_context["sex"].isin(["B", "BB"])
]

assert len(recovered_unresolved_rows) == 2
assert set(recovered_unresolved_rows["source_rowid"]) == set(
    unresolved_sex_rows["source_rowid"]
)

print("Complete race contexts containing unresolved sex codes")
display(unresolved_sex_race_context)

# Summarise the runner-sex vocabulary within each affected race.
unresolved_race_sex_summary = (
    unresolved_sex_race_context
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_name",
            "age_band",
            "sex_rest",
            "sex",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(runner_rows=("source_rowid", "count"))
    .sort_values(
        ["date", "course", "off", "runner_rows", "sex"],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)

print("Sex-code vocabulary within affected races")
display(unresolved_race_sex_summary)

Complete race contexts containing unresolved sex codes


,source_rowid,date,course,off,race_id,race_name,type,age_band,sex_rest,horse,age,sex,hg,num,pos,comment
0,448652,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Renfrew Street (GB),4,F,,8,1,Well into stride - soon led - headed after 4f ...
1,448651,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Adler (GER),3,C,,9,2,
2,448650,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Moonshiner (GER),4,C,b,1,3,
3,448649,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Iraklion (GER),5,H,,2,4,
4,448648,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Par Coeur (GER),3,BB,,10,5,
5,448661,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Kashmar (GER),4,F,,5,6,
6,448646,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Golden Gazelle (IRE),4,F,,6,7,
7,448644,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Sexy Juke (GER),3,F,,11,8,
8,448637,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Shadow Sadness (GER),5,H,,3,9,Held up towards rear of midfield - outpaced an...
9,448636,2017-10-15,Cologne (GER),1:35,687124,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,3yo+,,Super Ridge (FR),4,G,b,4,10,


Sex-code vocabulary within affected races


,date,course,off,race_name,age_band,sex_rest,sex,runner_rows
0,2017-10-15,Cologne (GER),1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,3yo+,,F,5
1,2017-10-15,Cologne (GER),1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,3yo+,,C,2
2,2017-10-15,Cologne (GER),1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,3yo+,,H,2
3,2017-10-15,Cologne (GER),1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,3yo+,,BB,1
4,2017-10-15,Cologne (GER),1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,3yo+,,G,1
5,2019-11-29,Gulfstream Park (USA),8:30,Starter Optional Claiming (Claimer) (2yo Filli...,2yo,F,F,10
6,2019-11-29,Gulfstream Park (USA),8:30,Starter Optional Claiming (Claimer) (2yo Filli...,2yo,F,B,1


## Stage 10 — Verify the two unresolved runner-sex values externally

The two unresolved source values are isolated one-row cases, and neither horse has another appearance in the source from which its sex can be established independently.

External evidence resolves both cases:

- `NB17-SEX-0002` — Par Coeur (GER) is recorded by Deutscher Galopp as a gelding. A published result additionally describes the horse as `b/br g`, meaning bay/brown gelding. The source value `BB` is therefore inconsistent with the verified runner sex and is consistent with colour information entering the `sex` field.
- `NB17-SEX-0003` — La Venezolana (VEN) is independently described as a bay mare, and the affected source race was restricted to two-year-old fillies. The source value `B` is therefore inconsistent with the verified runner sex and is consistent with the bay-colour code entering the `sex` field.

Published Racing Post guidance separately defines:

- `b` — bay;
- `br` — brown;
- `g` — gelding;
- `f` — filly;
- `m` — mare.

The evidence supports treating `B` and `BB` as two source-field contamination cases rather than additional runner-sex categories.

The immutable values must remain preserved. Any corrected sex values must be applied only through a governed downstream correction layer with the permanent verification identifiers.

In [9]:
# Reload the already-persisted Notebook 17 runner-sex exception evidence.
#
# These records were written successfully before the kernel restart.
# This cell does not append, overwrite or reconstruct them.
MANUAL_VERIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "manual_verifications.csv"
)

manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    dtype=str,
    keep_default_na=False,
)

sex_exception_verification_ids = {
    "NB17-SEX-0002",
    "NB17-SEX-0003",
}

reloaded_sex_exception_verifications = (
    manual_verifications.loc[
        manual_verifications["verification_id"].isin(
            sex_exception_verification_ids
        )
    ]
    .sort_values("verification_id")
    .reset_index(drop=True)
)

# Confirm that both permanent exception records exist exactly once.
assert len(reloaded_sex_exception_verifications) == 2
assert set(
    reloaded_sex_exception_verifications["verification_id"]
) == sex_exception_verification_ids
assert set(
    reloaded_sex_exception_verifications["verification_status"]
) == {"contradicted"}
assert set(
    reloaded_sex_exception_verifications["database_action"]
) == {"source_correction_candidate"}
assert set(
    reloaded_sex_exception_verifications["verified_value"]
) == {"G=gelding", "F=filly"}

print(
    "Notebook 17 runner-sex exceptions "
    "already persisted and successfully reloaded"
)

display(
    reloaded_sex_exception_verifications[
        [
            "verification_id",
            "source_date",
            "source_course",
            "source_horse",
            "raw_source_value",
            "verified_value",
            "verification_status",
            "confidence",
            "database_action",
        ]
    ]
)

Notebook 17 runner-sex exceptions already persisted and successfully reloaded


,verification_id,source_date,source_course,source_horse,raw_source_value,verified_value,verification_status,confidence,database_action
0,NB17-SEX-0002,2017-10-15,Cologne (GER),Par Coeur (GER),BB,G=gelding,contradicted,high,source_correction_candidate
1,NB17-SEX-0003,2019-11-29,Gulfstream Park (USA),La Venezolana (VEN),B,F=filly,contradicted,high,source_correction_candidate


## Stage 11 — Profile runner age against source-stated age bands

The source stores `age` as an integer on every governed runner row, with observed values from `1` to `31`.

Notebook 16 established that `age_band` contains source-stated race conditions. Parsed age-band bounds are useful contextual evidence, but they cannot be enforced universally as official eligibility rules without jurisdiction, authority and period evidence.

This stage therefore measures source-internal relationships only:

- runner-age coverage by observed value;
- the number of provisional races represented by each age;
- minimum and maximum race dates;
- coverage by race type;
- runner ages appearing below or above canonical parsed `age_band` bounds;
- the raw age-band forms associated with apparent contradictions.

An apparent contradiction will be flagged for review rather than automatically labelled a data error. Possible explanations include source defects, race-condition subtleties, international age conventions or incomplete interpretation of the race-level shorthand.

In [10]:
# Profile each observed runner age across rows, provisional races,
# source period and racing type.
age_distribution = pd.read_sql_query(
    f"""
    SELECT
        age AS raw_age,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            CAST(date AS TEXT) || '|' ||
            CAST(course AS TEXT) || '|' ||
            CAST(off AS TEXT)
        ) AS provisional_races,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(DISTINCT type) AS distinct_race_types
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY age
    ORDER BY age
    """,
    connection,
)

# Reconcile the complete age distribution to the governed runner population.
assert len(age_distribution) == 19
assert age_distribution["runner_rows"].sum() == EXPECTED_RUNNER_ROWS
assert age_distribution["raw_age"].min() == 1
assert age_distribution["raw_age"].max() == 31

print("Runner-age distribution")
display(age_distribution)

# Profile runner age by source race type without assuming that race-type
# labels have identical regulatory meaning across jurisdictions.
age_by_race_type = pd.read_sql_query(
    f"""
    SELECT
        type AS raw_race_type,
        age AS raw_age,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            CAST(date AS TEXT) || '|' ||
            CAST(course AS TEXT) || '|' ||
            CAST(off AS TEXT)
        ) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY type, age
    ORDER BY type, age
    """,
    connection,
)

assert age_by_race_type["runner_rows"].sum() == EXPECTED_RUNNER_ROWS

print("Runner age by source race type")
display(age_by_race_type)

# Parse only the canonical age-band forms already established by Notebook 16:
#
#   2yo
#   3yo+
#   3-5yo
#
# Other raw forms remain unparsed and are not included in the contradiction
# calculation.
canonical_age_band_relationships = pd.read_sql_query(
    f"""
    WITH parsed_age_bands AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            type,
            age_band,
            horse,
            age,
            CASE
                WHEN age_band GLOB '[0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 1) AS INTEGER)

                WHEN age_band GLOB '[0-9][0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 2) AS INTEGER)

                WHEN age_band GLOB '[0-9]yo+'
                THEN CAST(SUBSTR(age_band, 1, 1) AS INTEGER)

                WHEN age_band GLOB '[0-9][0-9]yo+'
                THEN CAST(SUBSTR(age_band, 1, 2) AS INTEGER)

                WHEN age_band GLOB '[0-9]-[0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 1) AS INTEGER)

                WHEN age_band GLOB '[0-9][0-9]-[0-9][0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 2) AS INTEGER)
            END AS parsed_min_age,
            CASE
                WHEN age_band GLOB '[0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 1) AS INTEGER)

                WHEN age_band GLOB '[0-9][0-9]yo'
                THEN CAST(SUBSTR(age_band, 1, 2) AS INTEGER)

                WHEN age_band GLOB '[0-9]-[0-9]yo'
                THEN CAST(SUBSTR(age_band, 3, 1) AS INTEGER)

                WHEN age_band GLOB '[0-9][0-9]-[0-9][0-9]yo'
                THEN CAST(SUBSTR(age_band, 4, 2) AS INTEGER)
            END AS parsed_max_age
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        *,
        CASE
            WHEN parsed_min_age IS NULL
            THEN 'unparsed_age_band'

            WHEN age < parsed_min_age
            THEN 'below_source_stated_minimum'

            WHEN parsed_max_age IS NOT NULL
             AND age > parsed_max_age
            THEN 'above_source_stated_maximum'

            ELSE 'within_parsed_source_bounds'
        END AS age_band_relationship
    FROM parsed_age_bands
    """,
    connection,
)

# Confirm that every governed runner row remains represented after the
# relationship classification.
assert len(canonical_age_band_relationships) == EXPECTED_RUNNER_ROWS

age_band_relationship_summary = (
    canonical_age_band_relationships
    .groupby("age_band_relationship", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        provisional_races=(
            "source_rowid",
            lambda rows: canonical_age_band_relationships.loc[
                rows.index,
                ["date", "course", "off"],
            ]
            .drop_duplicates()
            .shape[0],
        ),
    )
)

age_band_relationship_summary["runner_row_percentage"] = (
    age_band_relationship_summary["runner_rows"]
    / EXPECTED_RUNNER_ROWS
    * 100
)

assert (
    age_band_relationship_summary["runner_rows"].sum()
    == EXPECTED_RUNNER_ROWS
)

print("Runner-age relationship with canonical parsed age bands")
display(age_band_relationship_summary)

# Summarise the exact raw age-band and runner-age combinations associated
# with apparent lower- or upper-bound contradictions.
age_band_review_summary = (
    canonical_age_band_relationships.loc[
        canonical_age_band_relationships["age_band_relationship"].isin(
            [
                "below_source_stated_minimum",
                "above_source_stated_maximum",
            ]
        )
    ]
    .groupby(
        [
            "age_band_relationship",
            "age_band",
            "parsed_min_age",
            "parsed_max_age",
            "age",
            "type",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        provisional_races=(
            "source_rowid",
            lambda rows: canonical_age_band_relationships.loc[
                rows.index,
                ["date", "course", "off"],
            ]
            .drop_duplicates()
            .shape[0],
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "age_band_relationship",
            "age_band",
            "age",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

print("Apparent age-band contradictions requiring review")
display(age_band_review_summary)

Runner-age distribution


,raw_age,runner_rows,provisional_races,first_date,last_date,distinct_race_types
0,1,5,5,2018-04-30,2018-06-18,1
1,2,180383,19557,2015-01-01,2026-05-27,1
2,3,400774,66550,2015-01-01,2026-05-27,4
3,4,342278,96250,2015-01-01,2026-05-27,4
4,5,299403,104667,2015-01-01,2026-05-27,4
5,6,231753,100305,2015-01-01,2026-05-27,4
6,7,161945,84322,2015-01-01,2026-05-27,4
7,8,105400,62527,2015-01-01,2026-05-27,4
8,9,64630,42333,2015-01-01,2026-05-27,4
9,10,35998,25718,2015-01-01,2026-05-27,4


Runner age by source race type


,raw_race_type,raw_age,runner_rows,provisional_races
0,Chase,3,1186,158
1,Chase,4,7057,1566
2,Chase,5,11850,6499
3,Chase,6,28651,13723
4,Chase,7,38729,17128
5,Chase,8,34146,16228
6,Chase,9,25301,13339
7,Chase,10,16753,9912
8,Chase,11,9203,6335
9,Chase,12,4531,3545


Runner-age relationship with canonical parsed age bands


,age_band_relationship,runner_rows,provisional_races,runner_row_percentage
0,above_source_stated_maximum,840,112,0.045374
1,below_source_stated_minimum,118,71,0.006374
2,unparsed_age_band,159,13,0.008589
3,within_parsed_source_bounds,1850168,189000,99.939664


Apparent age-band contradictions requiring review


,age_band_relationship,age_band,parsed_min_age,parsed_max_age,age,type,runner_rows,provisional_races,first_date,last_date
0,above_source_stated_maximum,3yo,3.0,3.0,4,Flat,158,54,2015-03-01,2026-02-21
1,above_source_stated_maximum,3yo,3.0,3.0,5,Flat,126,44,2015-05-25,2025-12-18
2,above_source_stated_maximum,3yo,3.0,3.0,6,Flat,85,41,2015-05-25,2025-12-18
3,above_source_stated_maximum,2yo,2.0,2.0,3,Flat,84,16,2015-01-24,2025-06-22
4,above_source_stated_maximum,4yo,4.0,4.0,5,Flat,62,21,2015-01-17,2024-11-02
5,below_source_stated_minimum,3yo,3.0,3.0,2,Flat,60,29,2015-02-21,2025-12-21
6,above_source_stated_maximum,4yo,4.0,4.0,6,Flat,57,18,2015-01-17,2024-02-11
7,above_source_stated_maximum,3yo,3.0,3.0,7,Flat,35,26,2015-10-24,2025-12-18
8,above_source_stated_maximum,4yo,4.0,4.0,7,Flat,30,15,2015-01-17,2024-02-11
9,below_source_stated_minimum,4yo+,4.0,NaN,3,Flat,27,25,2015-06-13,2025-06-21


## Stage 12 — Locate age-band exceptions by governed jurisdiction and race pattern

The initial relationship check classifies 958 runner rows outside canonical parsed `age_band` bounds:

- 118 below a source-stated minimum;
- 840 above a source-stated maximum.

These flags do not yet establish that runner `age` is wrong. Some patterns involve many differently aged runners within the same race, which may instead indicate:

- an inaccurate or incomplete race-level `age_band`;
- a source extraction or classification defect;
- a jurisdiction-specific convention;
- a race condition not fully represented by the shorthand;
- an isolated runner-age error.

This stage joins the permanent governed course reference rather than deriving jurisdiction from course-text suffixes.

It will establish:

- complete course-reference join coverage;
- exception counts by governed jurisdiction;
- whether each affected race contains one exceptional runner or many;
- whether all runners in a race contradict the stated band;
- the most material affected race identities.

No correction or external verification will be authorised from this profiling alone.

In [11]:
# Reuse the governed Notebook 12 course-location implementation only for the
# 958 age-band exception rows.
#
# Applying its row-wise identity derivation to all 1.85 million runner rows is
# unnecessary and can exhaust the notebook kernel.

import sys

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from inside_rails.course_locations import (
    load_course_locations,
    merge_source_course_locations,
    unmatched_source_course_locations,
)

COURSE_LOCATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "course_locations.csv"
)

governed_course_locations = load_course_locations(
    COURSE_LOCATIONS_PATH
)

# Restrict the jurisdiction work to rows already identified as falling outside
# the canonical parsed source age-band bounds.
age_band_exception_rows = (
    canonical_age_band_relationships.loc[
        canonical_age_band_relationships[
            "age_band_relationship"
        ].isin(
            [
                "below_source_stated_minimum",
                "above_source_stated_maximum",
            ]
        )
    ]
    .copy()
)

assert len(age_band_exception_rows) == 958
assert age_band_exception_rows["source_rowid"].is_unique

# Derive governed course identities for the bounded exception population only.
exception_course_context = (
    age_band_exception_rows[
        [
            "source_rowid",
            "date",
            "course",
            "type",
            "race_name",
        ]
    ]
    .copy()
)

resolved_exception_course_context = merge_source_course_locations(
    exception_course_context,
    governed_course_locations,
    require_all_matches=False,
)

assert len(resolved_exception_course_context) == 958
assert resolved_exception_course_context["source_rowid"].is_unique

unmatched_exception_course_identities = (
    unmatched_source_course_locations(
        resolved_exception_course_context
    )
)

print("Unmatched governed course identities among age-band exceptions")
display(unmatched_exception_course_identities)

assert unmatched_exception_course_identities.empty

# Join the governed identities back to the 958 exception rows.
resolved_exception_jurisdiction = (
    resolved_exception_course_context[
        [
            "source_rowid",
            "candidate_course_label",
            "candidate_jurisdiction",
        ]
    ]
    .rename(
        columns={
            "candidate_course_label": "governed_course",
            "candidate_jurisdiction": "jurisdiction",
        }
    )
)

age_band_exception_rows = (
    age_band_exception_rows
    .merge(
        resolved_exception_jurisdiction,
        on="source_rowid",
        how="left",
        validate="one_to_one",
    )
)

assert len(age_band_exception_rows) == 958
assert age_band_exception_rows["jurisdiction"].notna().all()

print("Governed course-reference join for exception rows")
display(
    pd.DataFrame(
        {
            "measure": [
                "exception runner rows",
                "matched exception rows",
                "unmatched exception rows",
                "distinct source courses",
                "distinct governed courses",
                "distinct jurisdictions",
            ],
            "value": [
                len(age_band_exception_rows),
                int(
                    age_band_exception_rows[
                        "jurisdiction"
                    ].notna().sum()
                ),
                int(
                    age_band_exception_rows[
                        "jurisdiction"
                    ].isna().sum()
                ),
                age_band_exception_rows["course"].nunique(),
                age_band_exception_rows["governed_course"].nunique(),
                age_band_exception_rows["jurisdiction"].nunique(),
            ],
        }
    )
)

# Summarise exception rows by governed jurisdiction.
age_exceptions_by_jurisdiction = (
    age_band_exception_rows
    .groupby(
        ["jurisdiction", "age_band_relationship"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        provisional_races=(
            "source_rowid",
            lambda rows: age_band_exception_rows.loc[
                rows.index,
                ["date", "course", "off"],
            ]
            .drop_duplicates()
            .shape[0],
        ),
        distinct_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "jurisdiction",
            "age_band_relationship",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print("Age-band exceptions by governed jurisdiction")
display(age_exceptions_by_jurisdiction)

# Identify the affected provisional races.
race_key_columns = ["date", "course", "off"]

affected_race_keys = (
    age_band_exception_rows[race_key_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Use a MultiIndex membership test to select only runners from affected races.
# This avoids running course-identity derivation across the full source.
all_runner_race_index = pd.MultiIndex.from_frame(
    canonical_age_band_relationships[race_key_columns]
)

affected_race_index = pd.MultiIndex.from_frame(
    affected_race_keys
)

affected_race_mask = all_runner_race_index.isin(
    affected_race_index
)

affected_race_runner_rows = (
    canonical_age_band_relationships.loc[
        affected_race_mask
    ]
    .copy()
)

assert len(affected_race_runner_rows) >= 958

# Create one governed identity record per affected race from the already
# resolved exception rows.
affected_race_jurisdictions = (
    age_band_exception_rows[
        [
            *race_key_columns,
            "governed_course",
            "jurisdiction",
        ]
    ]
    .drop_duplicates()
)

if affected_race_jurisdictions.duplicated(
    subset=race_key_columns,
    keep=False,
).any():
    raise ValueError(
        "An affected provisional race resolved to multiple governed "
        "course identities."
    )

affected_race_runner_rows = (
    affected_race_runner_rows
    .merge(
        affected_race_jurisdictions,
        on=race_key_columns,
        how="left",
        validate="many_to_one",
    )
)

assert affected_race_runner_rows["jurisdiction"].notna().all()

race_group_columns = [
    "date",
    "course",
    "off",
    "race_id",
    "race_name",
    "type",
    "age_band",
    "parsed_min_age",
    "parsed_max_age",
    "governed_course",
    "jurisdiction",
]

race_population = (
    affected_race_runner_rows
    .groupby(
        race_group_columns,
        dropna=False,
        as_index=False,
    )
    .agg(
        total_runner_rows=("source_rowid", "count"),
        distinct_runner_ages=("age", "nunique"),
        minimum_runner_age=("age", "min"),
        maximum_runner_age=("age", "max"),
    )
)

race_exceptions = (
    age_band_exception_rows
    .groupby(
        race_group_columns,
        dropna=False,
        as_index=False,
    )
    .agg(
        exception_runner_rows=("source_rowid", "count"),
        below_minimum_rows=(
            "age_band_relationship",
            lambda values: (
                values == "below_source_stated_minimum"
            ).sum(),
        ),
        above_maximum_rows=(
            "age_band_relationship",
            lambda values: (
                values == "above_source_stated_maximum"
            ).sum(),
        ),
    )
)

age_exception_race_patterns = race_population.merge(
    race_exceptions,
    on=race_group_columns,
    how="inner",
    validate="one_to_one",
)

age_exception_race_patterns["exception_share"] = (
    age_exception_race_patterns["exception_runner_rows"]
    / age_exception_race_patterns["total_runner_rows"]
)

age_exception_race_patterns["race_exception_pattern"] = (
    age_exception_race_patterns.apply(
        lambda row: (
            "all_runner_rows_outside_bounds"
            if row["exception_runner_rows"]
            == row["total_runner_rows"]
            else (
                "single_runner_outside_bounds"
                if row["exception_runner_rows"] == 1
                else "multiple_but_not_all_outside_bounds"
            )
        ),
        axis=1,
    )
)

assert (
    age_exception_race_patterns[
        "exception_runner_rows"
    ].sum()
    == 958
)

race_pattern_summary = (
    age_exception_race_patterns
    .groupby("race_exception_pattern", as_index=False)
    .agg(
        provisional_races=("race_id", "count"),
        exception_runner_rows=("exception_runner_rows", "sum"),
        total_runner_rows=("total_runner_rows", "sum"),
    )
)

print("Exception pattern at provisional-race grain")
display(race_pattern_summary)

material_age_exception_races = (
    age_exception_race_patterns
    .sort_values(
        [
            "exception_runner_rows",
            "exception_share",
            "date",
            "course",
            "off",
        ],
        ascending=[False, False, True, True, True],
    )
    .reset_index(drop=True)
)

print("Material provisional races with age-band exceptions")
display(material_age_exception_races.head(30))

Unmatched governed course identities among age-band exceptions


,course,candidate_course_label,candidate_jurisdiction


Governed course-reference join for exception rows


,measure,value
0,exception runner rows,958
1,matched exception rows,958
2,unmatched exception rows,0
3,distinct source courses,83
4,distinct governed courses,78
5,distinct jurisdictions,24


Age-band exceptions by governed jurisdiction


,jurisdiction,age_band_relationship,runner_rows,provisional_races,distinct_courses,first_date,last_date
0,France,above_source_stated_maximum,238,25,9,2015-10-24,2022-04-07
1,Hong Kong,above_source_stated_maximum,138,12,2,2017-04-02,2025-04-02
2,Peru,above_source_stated_maximum,98,11,1,2021-06-26,2025-06-22
3,South Korea,above_source_stated_maximum,69,6,1,2022-09-04,2025-09-07
4,United States,above_source_stated_maximum,65,10,10,2015-01-17,2024-12-26
5,Australia,above_source_stated_maximum,35,9,7,2016-10-15,2026-02-21
6,South Africa,above_source_stated_maximum,29,3,2,2015-08-01,2023-06-03
7,Great Britain,below_source_stated_minimum,26,24,14,2015-06-20,2025-06-27
8,Canada,above_source_stated_maximum,23,6,2,2019-09-29,2024-10-06
9,Argentina,below_source_stated_minimum,21,2,1,2018-06-30,2018-06-30


Exception pattern at provisional-race grain


,race_exception_pattern,provisional_races,exception_runner_rows,total_runner_rows
0,all_runner_rows_outside_bounds,30,287,287
1,multiple_but_not_all_outside_bounds,74,592,885
2,single_runner_outside_bounds,79,79,867


Material provisional races with age-band exceptions


,date,course,off,race_id,race_name,type,age_band,parsed_min_age,parsed_max_age,governed_course,jurisdiction,total_runner_rows,distinct_runner_ages,minimum_runner_age,maximum_runner_age,exception_runner_rows,below_minimum_rows,above_maximum_rows,exception_share,race_exception_pattern
0,2015-08-01,Greyville (SAF),2:30,632817,Premiers Champion Stakes (2yo) (Turf),Flat,2yo,2.0,2.0,Greyville,South Africa,16,1,3,3,16,0,16,1.000000,all_runner_rows_outside_bounds
1,2024-06-23,Monterrico (PER),10:45,871606,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,2.0,2.0,Monterrico,Peru,16,3,3,5,16,0,16,1.000000,all_runner_rows_outside_bounds
2,2015-10-25,Saint-Cloud (FR),4:15,638007,Prix des Bords de Seine (Handicap) (4yo ) (Turf),Flat,4yo,4.0,4.0,Saint-Cloud,France,17,4,4,9,15,0,15,0.882353,multiple_but_not_all_outside_bounds
3,2015-07-04,Hipodromo Chile (CHI),9:52,632843,Premio Tanteo de Potrillos (2yo Colts) (Dirt),Flat,2yo,2.0,2.0,Hipodromo Chile,Chile,14,1,3,3,14,0,14,1.000000,all_runner_rows_outside_bounds
4,2017-05-31,Sha Tin (HK),12:45,677288,Silvermine Bay Handicap (3yo ) (All Weather T...,Flat,3yo,3.0,3.0,Sha Tin,Hong Kong,14,5,4,8,14,0,14,1.000000,all_runner_rows_outside_bounds
5,2021-01-01,Sha Tin (HK),8:35,775242,Cherry Handicap (3yo ) (Course B 2) (Turf),Flat,3yo,3.0,3.0,Sha Tin,Hong Kong,14,5,4,8,14,0,14,1.000000,all_runner_rows_outside_bounds
6,2025-06-22,Monterrico (PER),11:25,898593,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,2.0,2.0,Monterrico,Peru,14,3,3,6,14,0,14,1.000000,all_runner_rows_outside_bounds
7,2015-10-25,Saint-Cloud (FR),3:10,638003,Finale du Galop Tour Inter-Regional - GTI (Han...,Flat,4yo,4.0,4.0,Saint-Cloud,France,15,5,4,8,14,0,14,0.933333,multiple_but_not_all_outside_bounds
8,2024-09-08,Seoul (KOR),7:20,876301,Korea Sprint (3yo ) (Dirt),Flat,3yo,3.0,3.0,Seoul,South Korea,15,7,3,9,14,0,14,0.933333,multiple_but_not_all_outside_bounds
9,2016-09-12,Maisons-Laffitte (FR),12:47,659098,Prix des Aunettes (Handicap) (3yo ) (Turf),Flat,3yo,3.0,3.0,Maisons-Laffitte,France,16,5,3,7,14,0,14,0.875000,multiple_but_not_all_outside_bounds


## Stage 13 — Decompose the headgear field without over-interpreting it

The `hg` field is blank on 1,122,490 runner rows and populated on 728,795 rows with 60 distinct raw values.

Published Racing Post guidance verifies the principal component codes:

- `h` — hood;
- `b` — blinkers;
- `p` — cheekpieces;
- `t` — tongue-tie;
- `v` — visor;
- `e` — eye hood;
- `e/c` — eyecover;
- `e/s` — eyeshield.

It also explicitly documents `b1` and `b2` as first- and second-time blinkers.

The source additionally contains:

- concatenated combinations such as `tp`, `ht`, `tb` and `htp`;
- slash-bearing components such as `e/s`;
- trailing `1` values on several different combinations;
- rare components or combinations not covered by the initial reference.

This stage will decompose each raw value into observable components while preserving:

- the original raw string;
- component order;
- slash-bearing tokens;
- any trailing numeral;
- unresolved residue.

A successful decomposition does not by itself prove the semantic meaning of every combination or suffix.

In [12]:
# Decompose the complete populated headgear vocabulary while preserving each
# raw value exactly.
#
# The parser recognises only externally verified component codes. Any
# character sequence that cannot be consumed by those codes remains explicit
# unresolved residue rather than being guessed.

verified_headgear_components = [
    "e/c",
    "e/s",
    "h",
    "b",
    "p",
    "t",
    "v",
    "e",
]

def decompose_headgear_value(raw_value):
    """Parse verified headgear components, suffix and unresolved residue."""

    remaining = raw_value
    components = []

    # Preserve a single trailing numeral separately.
    use_suffix = ""

    if remaining.endswith(("1", "2")):
        use_suffix = remaining[-1]
        remaining = remaining[:-1]

    # Consume the remaining text from left to right, preferring slash-bearing
    # multi-character codes before one-character codes.
    while remaining:
        matched_component = None

        for component in verified_headgear_components:
            if remaining.startswith(component):
                matched_component = component
                break

        if matched_component is None:
            break

        components.append(matched_component)
        remaining = remaining[len(matched_component):]

    return pd.Series(
        {
            "parsed_components": "|".join(components),
            "component_count": len(components),
            "use_suffix": use_suffix,
            "unresolved_residue": remaining,
            "fully_decomposed": remaining == "",
        }
    )


headgear_decomposition = hg_vocabulary.copy()

parsed_headgear = headgear_decomposition["raw_hg"].apply(
    decompose_headgear_value
)

headgear_decomposition = pd.concat(
    [
        headgear_decomposition,
        parsed_headgear,
    ],
    axis=1,
)

# Confirm that all 60 populated raw values remain represented exactly once.
assert len(headgear_decomposition) == 60
assert headgear_decomposition["raw_hg"].is_unique
assert (
    headgear_decomposition["runner_rows"].sum()
    == 728_795
)

print("Complete headgear vocabulary decomposition")
display(
    headgear_decomposition[
        [
            "raw_hg",
            "runner_rows",
            "parsed_components",
            "component_count",
            "use_suffix",
            "unresolved_residue",
            "fully_decomposed",
        ]
    ]
)

# Summarise row coverage by decomposition status.
headgear_decomposition_summary = (
    headgear_decomposition
    .groupby("fully_decomposed", as_index=False)
    .agg(
        distinct_raw_values=("raw_hg", "nunique"),
        runner_rows=("runner_rows", "sum"),
    )
)

headgear_decomposition_summary[
    "populated_row_percentage"
] = (
    headgear_decomposition_summary["runner_rows"]
    / 728_795
    * 100
)

assert (
    headgear_decomposition_summary["runner_rows"].sum()
    == 728_795
)

print("Headgear decomposition coverage")
display(headgear_decomposition_summary)

# Show every unresolved raw value and residue.
unresolved_headgear_values = (
    headgear_decomposition.loc[
        ~headgear_decomposition["fully_decomposed"]
    ]
    .sort_values(
        ["runner_rows", "raw_hg"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("Headgear values with unresolved residue")
display(
    unresolved_headgear_values[
        [
            "raw_hg",
            "runner_rows",
            "parsed_components",
            "use_suffix",
            "unresolved_residue",
        ]
    ]
)

# Profile every observed trailing numeral by its unsuffixed base value.
headgear_suffix_profile = (
    headgear_decomposition.loc[
        headgear_decomposition["use_suffix"].ne("")
    ]
    .assign(
        unsuffixed_raw_value=lambda frame: (
            frame["raw_hg"].str[:-1]
        )
    )
    [
        [
            "raw_hg",
            "unsuffixed_raw_value",
            "use_suffix",
            "runner_rows",
            "parsed_components",
            "fully_decomposed",
        ]
    ]
    .sort_values(
        ["use_suffix", "runner_rows", "raw_hg"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

print("Observed headgear suffix profile")
display(headgear_suffix_profile)

Complete headgear vocabulary decomposition


,raw_hg,runner_rows,parsed_components,component_count,use_suffix,unresolved_residue,fully_decomposed
0,t,178727,t,1,,,True
1,p,173713,p,1,,,True
2,b,128859,b,1,,,True
3,h,62258,h,1,,,True
4,tp,52073,t|p,2,,,True
5,v,40585,v,1,,,True
6,tb,40299,t|b,2,,,True
7,ht,22190,h|t,2,,,True
8,tv,9729,t|v,2,,,True
9,e/s,2943,e/s,1,,,True


Headgear decomposition coverage


,fully_decomposed,distinct_raw_values,runner_rows,populated_row_percentage
0,False,3,9,0.001235
1,True,57,728786,99.998765


Headgear values with unresolved residue


,raw_hg,runner_rows,parsed_components,use_suffix,unresolved_residue
0,hc,5,h,,c
1,cvp,2,,,cvp
2,hct,2,h,,ct


Observed headgear suffix profile


,raw_hg,unsuffixed_raw_value,use_suffix,runner_rows,parsed_components,fully_decomposed
0,p1,p,1,1530,p,True
1,t1,t,1,1129,t,True
2,h1,h,1,800,h,True
3,b1,b,1,732,b,True
4,tp1,tp,1,588,t|p,True
5,v1,v,1,414,v,True
6,tb1,tb,1,284,t|b,True
7,ht1,ht,1,242,h|t,True
8,tv1,tv,1,131,t|v,True
9,hp1,hp,1,23,h|p,True


## Stage 14 — Inspect the nine headgear values containing unresolved `c`

Only three populated raw `hg` values fail decomposition:

- `hc` — five runner rows;
- `cvp` — two runner rows;
- `hct` — two runner rows.

All three contain `c`, which is not defined as headgear in the initial Racing Post reference.

This stage retrieves every affected runner and attaches governed jurisdiction context. It will test whether the values:

- occur within one jurisdiction or source period;
- recur for the same horse;
- coexist with otherwise standard equipment codes;
- appear to represent a jurisdiction-specific equipment code;
- are isolated transcription or field-contamination cases.

The raw values will remain unresolved until source repetition or external evidence establishes what `c` means.

In [13]:
# Retrieve every runner row carrying one of the three headgear values whose
# component c remains unresolved.
unresolved_headgear_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        age_band,
        sex_rest,
        horse,
        age,
        sex,
        hg,
        num,
        pos,
        trainer,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND hg IN ('hc', 'cvp', 'hct')
    ORDER BY date, course, off, horse, source_rowid
    """,
    connection,
)

assert len(unresolved_headgear_rows) == 9
assert set(unresolved_headgear_rows["hg"]) == {
    "hc",
    "cvp",
    "hct",
}

# Attach governed course and jurisdiction context using the bounded Notebook 12
# source-facing implementation. Only nine rows are processed here.
unresolved_headgear_course_context = (
    unresolved_headgear_rows[
        [
            "source_rowid",
            "date",
            "course",
            "type",
            "race_name",
        ]
    ]
    .copy()
)

resolved_unresolved_headgear_context = (
    merge_source_course_locations(
        unresolved_headgear_course_context,
        governed_course_locations,
        require_all_matches=False,
    )
)

unmatched_unresolved_headgear_courses = (
    unmatched_source_course_locations(
        resolved_unresolved_headgear_context
    )
)

assert unmatched_unresolved_headgear_courses.empty

unresolved_headgear_jurisdiction = (
    resolved_unresolved_headgear_context[
        [
            "source_rowid",
            "candidate_course_label",
            "candidate_jurisdiction",
        ]
    ]
    .rename(
        columns={
            "candidate_course_label": "governed_course",
            "candidate_jurisdiction": "jurisdiction",
        }
    )
)

unresolved_headgear_rows = (
    unresolved_headgear_rows
    .merge(
        unresolved_headgear_jurisdiction,
        on="source_rowid",
        how="left",
        validate="one_to_one",
    )
)

assert unresolved_headgear_rows["jurisdiction"].notna().all()

print("Runner rows containing unresolved headgear component c")
display(
    unresolved_headgear_rows[
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "type",
            "horse",
            "age",
            "sex",
            "hg",
            "trainer",
            "num",
            "pos",
            "governed_course",
            "jurisdiction",
            "comment",
        ]
    ]
)

# Summarise the unresolved values by governed jurisdiction and source period.
unresolved_headgear_context_summary = (
    unresolved_headgear_rows
    .groupby(
        ["hg", "jurisdiction"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_horses=("horse", "nunique"),
        distinct_courses=("course", "nunique"),
        distinct_trainers=("trainer", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["runner_rows", "hg", "jurisdiction"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print("Unresolved headgear values by governed jurisdiction")
display(unresolved_headgear_context_summary)

# Retrieve the complete source histories of all affected horse labels to see
# whether they appear elsewhere with a different or more readily interpreted
# headgear value.
unresolved_headgear_horses = (
    unresolved_headgear_rows["horse"]
    .drop_duplicates()
    .tolist()
)

headgear_horse_placeholders = ", ".join(
    "?" for _ in unresolved_headgear_horses
)

unresolved_headgear_horse_histories = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        horse,
        age,
        sex,
        hg,
        trainer,
        num,
        pos,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND horse IN ({headgear_horse_placeholders})
    ORDER BY horse, date, course, off, source_rowid
    """,
    connection,
    params=unresolved_headgear_horses,
)

assert set(
    unresolved_headgear_horse_histories["horse"]
) == set(unresolved_headgear_horses)

print("Complete source histories of affected horse labels")
display(unresolved_headgear_horse_histories)

# Summarise every headgear value recorded for each affected horse label.
unresolved_headgear_horse_summary = (
    unresolved_headgear_horse_histories
    .groupby(["horse", "hg"], as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_courses=("course", "nunique"),
    )
    .sort_values(
        ["horse", "runner_rows", "hg"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

print("Headgear histories of affected horse labels")
display(unresolved_headgear_horse_summary)

Runner rows containing unresolved headgear component c


,source_rowid,date,course,off,race_id,race_name,type,horse,age,sex,hg,trainer,num,pos,governed_course,jurisdiction,comment
0,505324,2018-03-09,Jebel Ali (UAE),1:00,696014,Roma Capanelle (Maiden) (Dirt),Flat,Al Abyad (USA),4,G,hct,A bin Harmash,1,5,Jebel Ali,United Arab Emirates,Mid-division - ran on same pace final 2 1/2f
1,616097,2018-10-12,Dundalk (AW) (IRE),5:30,713704,Irish Stallion Farms EBF (C & G) Median Auctio...,Flat,Aye Aye Captn (IRE),2,C,hc,Mrs John Harrington,1,9,Dundalk (AW),Ireland,Bit slowly away and raced towards rear - some ...
2,670502,2019-02-15,Dundalk (AW) (IRE),5:30,722750,Coral Free Bets For Lengths Maiden (Plus 10 Race),Flat,Eyeoweyou (IRE),3,G,hct,Jack W Davison,4,9,Dundalk (AW),Ireland,Slowly into stride and towards rear - raced ke...
3,673436,2019-02-22,Dundalk (AW) (IRE),6:00,723416,Irish Stallion Farms EBF Median Auction Maiden...,Flat,Eyeoweyou (IRE),3,G,hc,Jack W Davison,1,5,Dundalk (AW),Ireland,Prominent until soon settled behind leaders - ...
4,786381,2019-10-04,Dundalk (AW) (IRE),5:45,741541,Dundalkstadium.com Claiming Maiden,Flat,Silk Air (IRE),3,F,hc,J F Levins,4,9,Dundalk (AW),Ireland,Raced towards rear until progress over 1f out ...
5,837727,2020-02-01,Doha (QA),11:10,751820,H.E Sheikh Joaan Bin Hamad Al Tahni Trophy (Tu...,Flat,Shobrom (IRE),5,G,cvp,Ahmed Kobeissi,,5,Doha,Qatar,
6,837740,2020-02-01,Doha (QA),9:30,751822,H E Sheikh Joaan Bin Hamad Al Thani Trophy (Tu...,Flat,Connery (IRE),5,G,cvp,Ahmed Kobeissi,,2,Doha,Qatar,
7,990666,2021-04-08,Meydan (UAE),2:30,782016,Emirates Stakes Sponsored By Emirates Airline ...,Flat,Dignity Joy (USA),3,C,hc,M Al Mheiri,12,7,Meydan,United Arab Emirates,Mid-division - ran on same pace final 2f
8,1347987,2023-05-30,Redcar,4:20,839533,Racing TV Profits Returned To Racing Handicap,Flat,Humble Spark (IRE),3,G,hc,Jim Goldie,7,8,Redcar,Great Britain,Wore eye shield instead of declared eye cover ...


Unresolved headgear values by governed jurisdiction


,hg,jurisdiction,runner_rows,distinct_horses,distinct_courses,distinct_trainers,first_date,last_date
0,hc,Ireland,3,3,1,3,2018-10-12,2019-10-04
1,cvp,Qatar,2,2,1,1,2020-02-01,2020-02-01
2,hc,Great Britain,1,1,1,1,2023-05-30,2023-05-30
3,hc,United Arab Emirates,1,1,1,1,2021-04-08,2021-04-08
4,hct,Ireland,1,1,1,1,2019-02-15,2019-02-15
5,hct,United Arab Emirates,1,1,1,1,2018-03-09,2018-03-09


Complete source histories of affected horse labels


,source_rowid,date,course,off,race_id,race_name,type,horse,age,sex,hg,trainer,num,pos,comment
0,303709,2016-12-01,Meydan (UAE),3:40,664707,Longines La Grande Classique (Maiden) (Dirt),Flat,Al Abyad (USA),2,G,t,A bin Harmash,13,11,Slowly away - never near to challenge
1,310313,2016-12-17,Sharjah (UAE),11:00,665955,Longines Conquest Classic Moon Phase (Maiden) ...,Flat,Al Abyad (USA),2,G,t,A bin Harmash,14,3,Mid-division - ran on final 2f - never close t...
2,314837,2016-12-29,Meydan (UAE),2:30,666585,EGA Casthouse Trophy (Maiden) (Dirt),Flat,Al Abyad (USA),2,G,t,A bin Harmash,1,10,Tracked leaders until weakened final 3f
3,320752,2017-01-14,Meydan (UAE),12:00,667201,Al Naboodah Cargo Trophy (Maiden) (Turf),Flat,Al Abyad (USA),3,G,t,A bin Harmash,1,6,Slowly into stride - never near to challenge
4,463237,2017-11-11,Sharjah (UAE),11:30,688883,Sharjah Sports (Handicap) (Dirt),Flat,Al Abyad (USA),3,G,tb,A bin Harmash,4,5,Tracked leader - led 2 1/2f out - headed 1 1/2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,786381,2019-10-04,Dundalk (AW) (IRE),5:45,741541,Dundalkstadium.com Claiming Maiden,Flat,Silk Air (IRE),3,F,hc,J F Levins,4,9,Raced towards rear until progress over 1f out ...
114,871835,2020-08-08,Cork (IRE),4:50,764063,Thanks To All The Frontline Workers From Cork ...,Flat,Silk Air (IRE),4,F,t,J F Levins,21,8,Dwelt and went slightly right start - in touch...
115,888729,2020-09-10,Navan (IRE),4:50,766635,Abbey Road Handicap (Div II),Flat,Silk Air (IRE),4,F,t,J F Levins,2,11,Chased leaders - ridden to challenge under 3f ...
116,909733,2020-10-15,Curragh (IRE),3:15,769165,Equilux Works Or Your Money Back Handicap (Div...,Flat,Silk Air (IRE),4,F,t,J F Levins,5,9,Slowly away and pushed along briefly after sta...


Headgear histories of affected horse labels


,horse,hg,runner_rows,first_date,last_date,distinct_courses
0,Al Abyad (USA),t,5,2016-12-01,2017-12-16,3
1,Al Abyad (USA),tb,4,2017-11-11,2018-01-19,3
2,Al Abyad (USA),hct,1,2018-03-09,2018-03-09,1
3,Al Abyad (USA),ht,1,2018-03-02,2018-03-02,1
4,Aye Aye Captn (IRE),et,1,2019-04-29,2019-04-29,1
5,Aye Aye Captn (IRE),hc,1,2018-10-12,2018-10-12,1
6,Connery (IRE),,7,2017-06-03,2017-10-07,7
7,Connery (IRE),cvp,1,2020-02-01,2020-02-01,1
8,Dignity Joy (USA),v,8,2021-10-29,2022-12-03,3
9,Dignity Joy (USA),,3,2021-01-14,2022-01-07,2


## Stage 15 — Establish `c` as the source-specific eyecover component

The source does not contain the published Racing Post eyecover form `e/c`. Instead, nine runner rows contain `c` within:

- `hc`;
- `hct`;
- `cvp`.

The runner context provides direct supporting evidence. Humble Spark’s source comment states that the horse wore an eyeshield instead of the declared eye cover, while the row’s raw `hg` value is `hc`.

Interpreting `c` as the source-specific eyecover component makes all three forms coherent:

- `hc` — hood and eyecover;
- `hct` — hood, eyecover and tongue-tie;
- `cvp` — eyecover, visor and cheekpieces.

This is a source-specific normalisation rule rather than a claim that published racecards universally use standalone `c` for eyecover.

The immutable raw values remain preserved. A downstream equipment model may expose `c` as `eyecover` with explicit source lineage.

In [14]:
# Re-run the complete headgear decomposition with the source-specific c
# component interpreted as eyecover.
#
# The raw source value remains unchanged. The normalised component name makes
# clear that c and the published e/c notation represent the same equipment
# concept through different source encodings.

source_headgear_component_map = {
    "e/c": "eyecover",
    "e/s": "eyeshield",
    "h": "hood",
    "b": "blinkers",
    "p": "cheekpieces",
    "t": "tongue_tie",
    "v": "visor",
    "e": "eye_hood",
    "c": "eyecover",
}

# Prefer slash-bearing components before single-character components.
source_headgear_tokens = [
    "e/c",
    "e/s",
    "h",
    "b",
    "p",
    "t",
    "v",
    "e",
    "c",
]


def normalise_source_headgear(raw_value):
    """Decompose one source hg value into governed equipment components."""

    remaining = raw_value
    raw_components = []
    normalised_components = []

    # Preserve the source suffix independently from the equipment components.
    use_suffix = ""

    if remaining.endswith(("1", "2")):
        use_suffix = remaining[-1]
        remaining = remaining[:-1]

    # Consume the equipment string from left to right while retaining the
    # exact source component sequence.
    while remaining:
        matched_token = None

        for token in source_headgear_tokens:
            if remaining.startswith(token):
                matched_token = token
                break

        if matched_token is None:
            break

        raw_components.append(matched_token)
        normalised_components.append(
            source_headgear_component_map[matched_token]
        )

        remaining = remaining[len(matched_token):]

    return pd.Series(
        {
            "raw_components": "|".join(raw_components),
            "normalised_components": "|".join(
                normalised_components
            ),
            "component_count": len(raw_components),
            "use_suffix": use_suffix,
            "unresolved_residue": remaining,
            "fully_decomposed": remaining == "",
            "contains_source_eyecover_code": (
                "c" in raw_components
            ),
        }
    )


governed_headgear_vocabulary = hg_vocabulary.copy()

governed_headgear_parsing = (
    governed_headgear_vocabulary["raw_hg"]
    .apply(normalise_source_headgear)
)

governed_headgear_vocabulary = pd.concat(
    [
        governed_headgear_vocabulary,
        governed_headgear_parsing,
    ],
    axis=1,
)

# Every populated source value must now decompose completely.
assert len(governed_headgear_vocabulary) == 60
assert governed_headgear_vocabulary["raw_hg"].is_unique
assert governed_headgear_vocabulary["fully_decomposed"].all()
assert (
    governed_headgear_vocabulary["runner_rows"].sum()
    == 728_795
)

print("Governed complete headgear vocabulary")
display(
    governed_headgear_vocabulary[
        [
            "raw_hg",
            "runner_rows",
            "raw_components",
            "normalised_components",
            "component_count",
            "use_suffix",
            "contains_source_eyecover_code",
            "fully_decomposed",
        ]
    ]
)

# Isolate the nine rows whose interpretation depends on the source-specific
# c-to-eyecover rule.
source_eyecover_code_profile = (
    governed_headgear_vocabulary.loc[
        governed_headgear_vocabulary[
            "contains_source_eyecover_code"
        ]
    ]
    [
        [
            "raw_hg",
            "runner_rows",
            "raw_components",
            "normalised_components",
            "use_suffix",
        ]
    ]
    .sort_values(
        ["runner_rows", "raw_hg"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

assert set(source_eyecover_code_profile["raw_hg"]) == {
    "hc",
    "hct",
    "cvp",
}
assert source_eyecover_code_profile["runner_rows"].sum() == 9

print("Source-specific c-to-eyecover mappings")
display(source_eyecover_code_profile)

# Summarise component-level row exposure across the populated source.
#
# A runner carrying multiple pieces of equipment contributes once to each
# component, so these counts are not expected to sum to the populated-row
# total.
headgear_component_exposure_records = []

for row in governed_headgear_vocabulary.itertuples(index=False):
    components = row.normalised_components.split("|")

    for component in components:
        headgear_component_exposure_records.append(
            {
                "normalised_component": component,
                "raw_hg": row.raw_hg,
                "runner_rows": row.runner_rows,
            }
        )

headgear_component_exposure = (
    pd.DataFrame(headgear_component_exposure_records)
    .groupby("normalised_component", as_index=False)
    .agg(
        runner_rows=("runner_rows", "sum"),
        distinct_raw_values=("raw_hg", "nunique"),
    )
    .sort_values(
        ["runner_rows", "normalised_component"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("Normalised headgear component exposure")
display(headgear_component_exposure)

Governed complete headgear vocabulary


,raw_hg,runner_rows,raw_components,normalised_components,component_count,use_suffix,contains_source_eyecover_code,fully_decomposed
0,t,178727,t,tongue_tie,1,,False,True
1,p,173713,p,cheekpieces,1,,False,True
2,b,128859,b,blinkers,1,,False,True
3,h,62258,h,hood,1,,False,True
4,tp,52073,t|p,tongue_tie|cheekpieces,2,,False,True
5,v,40585,v,visor,1,,False,True
6,tb,40299,t|b,tongue_tie|blinkers,2,,False,True
7,ht,22190,h|t,hood|tongue_tie,2,,False,True
8,tv,9729,t|v,tongue_tie|visor,2,,False,True
9,e/s,2943,e/s,eyeshield,1,,False,True


Source-specific c-to-eyecover mappings


,raw_hg,runner_rows,raw_components,normalised_components,use_suffix
0,hc,5,h|c,hood|eyecover,
1,cvp,2,c|v|p,eyecover|visor|cheekpieces,
2,hct,2,h|c|t,hood|eyecover|tongue_tie,


Normalised headgear component exposure


,normalised_component,runner_rows,distinct_raw_values
0,tongue_tie,310116,29
1,cheekpieces,231288,20
2,blinkers,174852,20
3,hood,94177,27
4,visor,51382,11
5,eye_hood,4257,19
6,eyeshield,2999,3
7,eyecover,9,3


## Stage 16 — Test the observed trailing `1` against runner equipment histories

The source contains 22 suffixed raw `hg` values, all ending in `1`. No populated value ends in `2`.

Published Racing Post guidance explicitly verifies:

- `b1` — first-time blinkers;
- `b2` — second-time blinkers.

The source also applies `1` to other equipment and combinations, including:

- `p1`;
- `t1`;
- `h1`;
- `v1`;
- `tp1`;
- `e/s1`;
- multi-component combinations.

It would be unsafe to assume from `b1` alone that every trailing `1` universally means first-time use.

This stage tests the source behaviour directly by comparing every suffixed runner row with earlier appearances of the same source horse label.

The test can establish whether:

- the exact unsuffixed equipment combination appeared previously;
- any component in the suffixed combination appeared previously;
- the suffix usually marks the first recorded source appearance of the equipment;
- incomplete horse histories prevent a definitive conclusion.

A first appearance in this database is not automatically the horse’s first official use. The result will therefore be described as source-recorded history rather than universal equipment history.

In [16]:
# Retrieve every runner row whose populated headgear value ends in 1.
#
# All observed suffixed values use 1; there are no populated source values
# ending in 2.
suffixed_headgear_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        horse,
        age,
        sex,
        hg,
        trainer,
        num,
        pos
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND hg <> ''
      AND SUBSTR(hg, -1, 1) = '1'
    ORDER BY horse, date, course, off, source_rowid
    """,
    connection,
)

expected_suffixed_rows = int(
    headgear_suffix_profile["runner_rows"].sum()
)

assert len(suffixed_headgear_rows) == expected_suffixed_rows
assert suffixed_headgear_rows["hg"].str.endswith("1").all()

# Derive the unsuffixed raw value and its governed component set.
suffixed_headgear_rows["unsuffixed_hg"] = (
    suffixed_headgear_rows["hg"].str[:-1]
)

suffixed_component_lookup = (
    governed_headgear_vocabulary[
        [
            "raw_hg",
            "normalised_components",
        ]
    ]
    .rename(columns={"raw_hg": "unsuffixed_hg"})
)

suffixed_headgear_rows = (
    suffixed_headgear_rows
    .merge(
        suffixed_component_lookup,
        on="unsuffixed_hg",
        how="left",
        validate="many_to_one",
    )
)

assert suffixed_headgear_rows[
    "normalised_components"
].notna().all()

# Load all source appearances for horses that ever carry a suffixed value.
suffixed_horses = (
    suffixed_headgear_rows["horse"]
    .drop_duplicates()
    .tolist()
)

suffixed_horse_placeholders = ", ".join(
    "?" for _ in suffixed_horses
)

suffixed_horse_histories = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        hg
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND horse IN ({suffixed_horse_placeholders})
    ORDER BY horse, date, course, off, source_rowid
    """,
    connection,
    params=suffixed_horses,
)

# Parse each historical populated hg value into a set of normalised components.
history_component_lookup = (
    governed_headgear_vocabulary[
        [
            "raw_hg",
            "normalised_components",
        ]
    ]
    .set_index("raw_hg")["normalised_components"]
    .to_dict()
)

def component_set(raw_hg):
    """Return normalised components for one populated historical hg value."""

    if raw_hg == "":
        return set()

    components = history_component_lookup.get(raw_hg)

    if components is None:
        raise ValueError(
            f"Historical hg value was not governed: {raw_hg!r}"
        )

    return set(components.split("|"))


history_by_horse = {
    horse: frame.copy()
    for horse, frame in suffixed_horse_histories.groupby(
        "horse",
        sort=False,
    )
}

suffix_history_records = []

for row in suffixed_headgear_rows.itertuples(index=False):
    horse_history = history_by_horse[row.horse]

    # source_rowid provides an exact boundary around the current source row.
    earlier_history = horse_history.loc[
        horse_history["source_rowid"] < row.source_rowid
    ]

    target_components = set(
        row.normalised_components.split("|")
    )

    prior_populated_values = (
        earlier_history.loc[
            earlier_history["hg"].ne(""),
            "hg",
        ]
        .tolist()
    )

    prior_component_sets = [
        component_set(raw_hg)
        for raw_hg in prior_populated_values
    ]

    exact_base_seen_earlier = (
        row.unsuffixed_hg in prior_populated_values
    )

    any_target_component_seen_earlier = any(
        bool(target_components.intersection(previous_components))
        for previous_components in prior_component_sets
    )

    all_target_components_seen_earlier = (
        bool(prior_component_sets)
        and all(
            any(
                component in previous_components
                for previous_components in prior_component_sets
            )
            for component in target_components
        )
    )

    suffix_history_records.append(
        {
            "source_rowid": row.source_rowid,
            "date": row.date,
            "course": row.course,
            "off": row.off,
            "horse": row.horse,
            "raw_hg": row.hg,
            "unsuffixed_hg": row.unsuffixed_hg,
            "normalised_components": row.normalised_components,
            "earlier_source_rows": len(earlier_history),
            "earlier_populated_hg_rows": len(
                prior_populated_values
            ),
            "exact_base_seen_earlier": exact_base_seen_earlier,
            "any_target_component_seen_earlier": (
                any_target_component_seen_earlier
            ),
            "all_target_components_seen_earlier": (
                all_target_components_seen_earlier
            ),
        }
    )

suffixed_headgear_history_test = pd.DataFrame(
    suffix_history_records
)

assert len(suffixed_headgear_history_test) == expected_suffixed_rows
assert suffixed_headgear_history_test["source_rowid"].is_unique

# Classify each suffixed row from the observable source history.
def classify_suffix_history(row):
    """Describe what the prior source history establishes."""

    if row["earlier_source_rows"] == 0:
        return "no_earlier_source_history"

    if row["exact_base_seen_earlier"]:
        return "exact_unsuffixed_value_seen_earlier"

    if row["all_target_components_seen_earlier"]:
        return "all_components_seen_earlier_in_other_forms"

    if row["any_target_component_seen_earlier"]:
        return "some_components_seen_earlier"

    return "no_target_component_seen_earlier"


suffixed_headgear_history_test[
    "source_history_status"
] = suffixed_headgear_history_test.apply(
    classify_suffix_history,
    axis=1,
)

suffix_history_summary = (
    suffixed_headgear_history_test
    .groupby("source_history_status", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_horses=("horse", "nunique"),
        distinct_suffixed_values=("raw_hg", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["runner_rows", "source_history_status"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

assert (
    suffix_history_summary["runner_rows"].sum()
    == expected_suffixed_rows
)

print("Trailing-1 behaviour against prior source history")
display(suffix_history_summary)

# Profile the history result separately for each raw suffixed value.
suffix_history_by_value = (
    suffixed_headgear_history_test
    .groupby(
        ["raw_hg", "source_history_status"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_horses=("horse", "nunique"),
    )
    .sort_values(
        ["raw_hg", "runner_rows", "source_history_status"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

print("Trailing-1 history result by raw value")
display(suffix_history_by_value)

# Show the strongest contradictions to a simple first-recorded-use reading:
# rows where the exact unsuffixed value had already appeared for that horse.
prior_exact_base_examples = (
    suffixed_headgear_history_test.loc[
        suffixed_headgear_history_test[
            "exact_base_seen_earlier"
        ]
    ]
    .sort_values(
        ["raw_hg", "horse", "date", "course", "off"],
    )
    .reset_index(drop=True)
)

print("Suffixed rows whose exact unsuffixed value appeared earlier")
display(prior_exact_base_examples.head(50))

Trailing-1 behaviour against prior source history


,source_history_status,runner_rows,distinct_horses,distinct_suffixed_values,first_date,last_date
0,no_target_component_seen_earlier,4353,3949,15,2025-10-15,2026-05-27
1,some_components_seen_earlier,987,916,15,2025-10-15,2026-05-27
2,no_earlier_source_history,450,450,11,2025-10-15,2026-05-27
3,all_components_seen_earlier_in_other_forms,139,135,8,2025-10-15,2026-05-27
4,exact_unsuffixed_value_seen_earlier,3,3,2,2026-05-03,2026-05-14


Trailing-1 history result by raw value


,raw_hg,source_history_status,runner_rows,distinct_horses
0,b1,no_target_component_seen_earlier,691,691
1,b1,no_earlier_source_history,41,41
2,bp1,some_components_seen_earlier,1,1
3,e/s1,no_target_component_seen_earlier,22,22
4,eb1,no_target_component_seen_earlier,4,4
5,et1,some_components_seen_earlier,4,4
6,et1,no_target_component_seen_earlier,2,2
7,etb1,all_components_seen_earlier_in_other_forms,1,1
8,etb1,some_components_seen_earlier,1,1
9,h1,no_target_component_seen_earlier,609,609


Suffixed rows whose exact unsuffixed value appeared earlier


,source_rowid,date,course,off,horse,raw_hg,unsuffixed_hg,normalised_components,earlier_source_rows,earlier_populated_hg_rows,exact_base_seen_earlier,any_target_component_seen_earlier,all_target_components_seen_earlier,source_history_status
0,1838705,2026-05-03,Sligo,15:22,Hobart (IRE),p1,p,cheekpieces,16,2,True,True,True,exact_unsuffixed_value_seen_earlier
1,1843240,2026-05-12,Killarney,19:10,Mount Eden (IRE),p1,p,cheekpieces,4,1,True,True,True,exact_unsuffixed_value_seen_earlier
2,1844279,2026-05-14,Perth,13:58,Breaking Ground (IRE),t1,t,tongue_tie,21,13,True,True,True,exact_unsuffixed_value_seen_earlier


## Stage 17 — Establish the temporal introduction of headgear suffixes

Every observed trailing-`1` value occurs from 15 October 2025 onward, despite the source beginning in 2015.

This strongly suggests a change in source presentation or collection rather than a stable field convention available throughout the full period.

This stage will measure:

- the first and last appearance of every raw `hg` value;
- suffixed and unsuffixed coverage by year and month;
- whether unsuffixed equipment values continue after suffixes appear;
- whether the source ever records a trailing `2`;
- whether suffix coverage is concentrated by jurisdiction or race type.

The suffix will remain a source-presented declaration. Earlier source history cannot safely validate or invalidate the declaration because:

- the field convention may have changed;
- historical equipment reporting may be incomplete;
- first-time status may apply to one component within a combination;
- isolated source inconsistencies may exist.

In [17]:
# Profile the temporal introduction and coverage of headgear suffixes.
#
# This tests whether trailing numerals are a stable convention throughout the
# source or a recently introduced presentation feature.

headgear_temporal_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        type,
        hg
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND hg <> ''
    ORDER BY date, source_rowid
    """,
    connection,
)

assert len(headgear_temporal_rows) == 728_795

headgear_temporal_rows["date"] = pd.to_datetime(
    headgear_temporal_rows["date"],
    errors="raise",
)

headgear_temporal_rows["year"] = (
    headgear_temporal_rows["date"].dt.year
)

headgear_temporal_rows["year_month"] = (
    headgear_temporal_rows["date"]
    .dt.to_period("M")
    .astype(str)
)

headgear_temporal_rows["trailing_character"] = (
    headgear_temporal_rows["hg"].str[-1]
)

headgear_temporal_rows["suffix_status"] = (
    headgear_temporal_rows["trailing_character"]
    .map(
        {
            "1": "trailing_1",
            "2": "trailing_2",
        }
    )
    .fillna("no_trailing_numeral")
)

# Confirm the complete observed numeral vocabulary.
observed_numeral_suffixes = sorted(
    headgear_temporal_rows.loc[
        headgear_temporal_rows[
            "trailing_character"
        ].str.fullmatch(r"\d"),
        "trailing_character",
    ].unique()
)

print("Observed numeric headgear suffixes")
display(
    pd.DataFrame(
        {
            "observed_numeric_suffix": observed_numeral_suffixes,
        }
    )
)

# Profile the lifetime of every populated raw headgear value.
headgear_value_periods = (
    headgear_temporal_rows
    .groupby("hg", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

headgear_value_periods["has_trailing_1"] = (
    headgear_value_periods["hg"].str.endswith("1")
)

headgear_value_periods = (
    headgear_value_periods
    .sort_values(
        ["has_trailing_1", "runner_rows", "hg"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print("Observed period of every populated headgear value")
display(headgear_value_periods)

# Summarise suffixed and unsuffixed coverage by calendar year.
headgear_suffix_by_year = (
    headgear_temporal_rows
    .groupby(
        ["year", "suffix_status"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_raw_values=("hg", "nunique"),
        distinct_courses=("course", "nunique"),
        distinct_race_types=("type", "nunique"),
    )
    .sort_values(["year", "suffix_status"])
    .reset_index(drop=True)
)

print("Headgear suffix coverage by year")
display(headgear_suffix_by_year)

# Show monthly coverage from the first observed trailing-1 row onward.
first_trailing_1_date = (
    headgear_temporal_rows.loc[
        headgear_temporal_rows[
            "suffix_status"
        ].eq("trailing_1"),
        "date",
    ]
    .min()
)

assert first_trailing_1_date == pd.Timestamp("2025-10-15")

suffix_era_monthly_profile = (
    headgear_temporal_rows.loc[
        headgear_temporal_rows["date"]
        >= first_trailing_1_date
    ]
    .groupby(
        ["year_month", "suffix_status"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_raw_values=("hg", "nunique"),
        distinct_courses=("course", "nunique"),
    )
    .sort_values(["year_month", "suffix_status"])
    .reset_index(drop=True)
)

print("Monthly headgear coverage from suffix introduction")
display(suffix_era_monthly_profile)

# Derive governed jurisdiction only for the bounded suffixed population.
suffixed_jurisdiction_context = (
    suffixed_headgear_rows[
        [
            "source_rowid",
            "date",
            "course",
            "type",
            "race_name",
        ]
    ]
    .copy()
)

resolved_suffixed_jurisdictions = (
    merge_source_course_locations(
        suffixed_jurisdiction_context,
        governed_course_locations,
        require_all_matches=False,
    )
)

assert unmatched_source_course_locations(
    resolved_suffixed_jurisdictions
).empty

suffix_jurisdiction_profile = (
    resolved_suffixed_jurisdictions
    .groupby(
        "candidate_jurisdiction",
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["runner_rows", "candidate_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

assert suffix_jurisdiction_profile[
    "runner_rows"
].sum() == expected_suffixed_rows

print("Trailing-1 rows by governed jurisdiction")
display(suffix_jurisdiction_profile)

Observed numeric headgear suffixes


,observed_numeric_suffix
0,1


Observed period of every populated headgear value


,hg,runner_rows,first_date,last_date,has_trailing_1
0,p1,1530,2025-10-15,2026-05-27,True
1,t1,1129,2025-10-15,2026-05-27,True
2,h1,800,2025-10-15,2026-05-27,True
3,b1,732,2025-10-15,2026-05-27,True
4,tp1,588,2025-10-15,2026-05-27,True
5,v1,414,2025-10-15,2026-05-26,True
6,tb1,284,2025-10-15,2026-05-27,True
7,ht1,242,2025-10-15,2026-05-27,True
8,tv1,131,2025-10-15,2026-05-27,True
9,hp1,23,2025-10-16,2026-05-23,True


Headgear suffix coverage by year


,year,suffix_status,runner_rows,distinct_raw_values,distinct_courses,distinct_race_types
0,2015,no_trailing_numeral,52710,26,275,4
1,2016,no_trailing_numeral,54494,25,236,4
2,2017,no_trailing_numeral,63545,27,241,4
3,2018,no_trailing_numeral,63972,28,247,4
4,2019,no_trailing_numeral,63748,28,255,4
5,2020,no_trailing_numeral,45935,28,210,4
6,2021,no_trailing_numeral,70050,26,228,4
7,2022,no_trailing_numeral,70657,24,238,4
8,2023,no_trailing_numeral,65496,28,208,4
9,2024,no_trailing_numeral,74128,26,220,4


Monthly headgear coverage from suffix introduction


,year_month,suffix_status,runner_rows,distinct_raw_values,distinct_courses
0,2025-10,no_trailing_numeral,3645,17,81
1,2025-10,trailing_1,547,11,64
2,2025-11,no_trailing_numeral,5784,19,90
3,2025-11,trailing_1,829,15,71
4,2025-12,no_trailing_numeral,5057,17,76
5,2025-12,trailing_1,785,14,63
6,2026-01,no_trailing_numeral,4978,19,72
7,2026-01,trailing_1,645,13,52
8,2026-02,no_trailing_numeral,4371,20,74
9,2026-02,trailing_1,635,15,50


Trailing-1 rows by governed jurisdiction


,candidate_jurisdiction,runner_rows,distinct_courses,first_date,last_date
0,Great Britain,3900,63,2025-10-15,2026-05-27
1,Ireland,1660,23,2025-10-15,2026-05-27
2,Hong Kong,199,2,2025-10-15,2026-03-08
3,France,78,7,2025-10-15,2026-03-19
4,Australia,40,8,2025-10-18,2025-12-06
5,Saudi Arabia,25,1,2026-02-14,2026-02-14
6,United States,14,5,2025-10-25,2025-12-28
7,Bahrain,5,1,2025-11-14,2026-02-19
8,New Zealand,5,3,2025-11-08,2026-01-03
9,Switzerland,3,1,2026-02-22,2026-02-22


## Stage 17 — Establish the temporal introduction of headgear suffixes

Every observed trailing-`1` value occurs from 15 October 2025 onward, despite the source beginning in 2015.

This strongly suggests a change in source presentation or collection rather than a stable field convention available throughout the full period.

This stage will measure:

- the first and last appearance of every raw `hg` value;
- suffixed and unsuffixed coverage by year and month;
- whether unsuffixed equipment values continue after suffixes appear;
- whether the source ever records a trailing `2`;
- whether suffix coverage is concentrated by jurisdiction or race type.

The suffix will remain a source-presented declaration. Earlier source history cannot safely validate or invalidate the declaration because:

- the field convention may have changed;
- historical equipment reporting may be incomplete;
- first-time status may apply to one component within a combination;
- isolated source inconsistencies may exist.

In [18]:
# Profile the temporal introduction and coverage of headgear suffixes.
#
# This tests whether trailing numerals are a stable convention throughout the
# source or a recently introduced presentation feature.

headgear_temporal_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        type,
        hg
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND hg <> ''
    ORDER BY date, source_rowid
    """,
    connection,
)

assert len(headgear_temporal_rows) == 728_795

headgear_temporal_rows["date"] = pd.to_datetime(
    headgear_temporal_rows["date"],
    errors="raise",
)

headgear_temporal_rows["year"] = (
    headgear_temporal_rows["date"].dt.year
)

headgear_temporal_rows["year_month"] = (
    headgear_temporal_rows["date"]
    .dt.to_period("M")
    .astype(str)
)

headgear_temporal_rows["trailing_character"] = (
    headgear_temporal_rows["hg"].str[-1]
)

headgear_temporal_rows["suffix_status"] = (
    headgear_temporal_rows["trailing_character"]
    .map(
        {
            "1": "trailing_1",
            "2": "trailing_2",
        }
    )
    .fillna("no_trailing_numeral")
)

# Confirm the complete observed numeral vocabulary.
observed_numeral_suffixes = sorted(
    headgear_temporal_rows.loc[
        headgear_temporal_rows[
            "trailing_character"
        ].str.fullmatch(r"\d"),
        "trailing_character",
    ].unique()
)

print("Observed numeric headgear suffixes")
display(
    pd.DataFrame(
        {
            "observed_numeric_suffix": observed_numeral_suffixes,
        }
    )
)

# Profile the lifetime of every populated raw headgear value.
headgear_value_periods = (
    headgear_temporal_rows
    .groupby("hg", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

headgear_value_periods["has_trailing_1"] = (
    headgear_value_periods["hg"].str.endswith("1")
)

headgear_value_periods = (
    headgear_value_periods
    .sort_values(
        ["has_trailing_1", "runner_rows", "hg"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print("Observed period of every populated headgear value")
display(headgear_value_periods)

# Summarise suffixed and unsuffixed coverage by calendar year.
headgear_suffix_by_year = (
    headgear_temporal_rows
    .groupby(
        ["year", "suffix_status"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_raw_values=("hg", "nunique"),
        distinct_courses=("course", "nunique"),
        distinct_race_types=("type", "nunique"),
    )
    .sort_values(["year", "suffix_status"])
    .reset_index(drop=True)
)

print("Headgear suffix coverage by year")
display(headgear_suffix_by_year)

# Show monthly coverage from the first observed trailing-1 row onward.
first_trailing_1_date = (
    headgear_temporal_rows.loc[
        headgear_temporal_rows[
            "suffix_status"
        ].eq("trailing_1"),
        "date",
    ]
    .min()
)

assert first_trailing_1_date == pd.Timestamp("2025-10-15")

suffix_era_monthly_profile = (
    headgear_temporal_rows.loc[
        headgear_temporal_rows["date"]
        >= first_trailing_1_date
    ]
    .groupby(
        ["year_month", "suffix_status"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_raw_values=("hg", "nunique"),
        distinct_courses=("course", "nunique"),
    )
    .sort_values(["year_month", "suffix_status"])
    .reset_index(drop=True)
)

print("Monthly headgear coverage from suffix introduction")
display(suffix_era_monthly_profile)

# Derive governed jurisdiction only for the bounded suffixed population.
suffixed_jurisdiction_context = (
    suffixed_headgear_rows[
        [
            "source_rowid",
            "date",
            "course",
            "type",
            "race_name",
        ]
    ]
    .copy()
)

resolved_suffixed_jurisdictions = (
    merge_source_course_locations(
        suffixed_jurisdiction_context,
        governed_course_locations,
        require_all_matches=False,
    )
)

assert unmatched_source_course_locations(
    resolved_suffixed_jurisdictions
).empty

suffix_jurisdiction_profile = (
    resolved_suffixed_jurisdictions
    .groupby(
        "candidate_jurisdiction",
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        distinct_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["runner_rows", "candidate_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

assert suffix_jurisdiction_profile[
    "runner_rows"
].sum() == expected_suffixed_rows

print("Trailing-1 rows by governed jurisdiction")
display(suffix_jurisdiction_profile)

Observed numeric headgear suffixes


,observed_numeric_suffix
0,1


Observed period of every populated headgear value


,hg,runner_rows,first_date,last_date,has_trailing_1
0,p1,1530,2025-10-15,2026-05-27,True
1,t1,1129,2025-10-15,2026-05-27,True
2,h1,800,2025-10-15,2026-05-27,True
3,b1,732,2025-10-15,2026-05-27,True
4,tp1,588,2025-10-15,2026-05-27,True
5,v1,414,2025-10-15,2026-05-26,True
6,tb1,284,2025-10-15,2026-05-27,True
7,ht1,242,2025-10-15,2026-05-27,True
8,tv1,131,2025-10-15,2026-05-27,True
9,hp1,23,2025-10-16,2026-05-23,True


Headgear suffix coverage by year


,year,suffix_status,runner_rows,distinct_raw_values,distinct_courses,distinct_race_types
0,2015,no_trailing_numeral,52710,26,275,4
1,2016,no_trailing_numeral,54494,25,236,4
2,2017,no_trailing_numeral,63545,27,241,4
3,2018,no_trailing_numeral,63972,28,247,4
4,2019,no_trailing_numeral,63748,28,255,4
5,2020,no_trailing_numeral,45935,28,210,4
6,2021,no_trailing_numeral,70050,26,228,4
7,2022,no_trailing_numeral,70657,24,238,4
8,2023,no_trailing_numeral,65496,28,208,4
9,2024,no_trailing_numeral,74128,26,220,4


Monthly headgear coverage from suffix introduction


,year_month,suffix_status,runner_rows,distinct_raw_values,distinct_courses
0,2025-10,no_trailing_numeral,3645,17,81
1,2025-10,trailing_1,547,11,64
2,2025-11,no_trailing_numeral,5784,19,90
3,2025-11,trailing_1,829,15,71
4,2025-12,no_trailing_numeral,5057,17,76
5,2025-12,trailing_1,785,14,63
6,2026-01,no_trailing_numeral,4978,19,72
7,2026-01,trailing_1,645,13,52
8,2026-02,no_trailing_numeral,4371,20,74
9,2026-02,trailing_1,635,15,50


Trailing-1 rows by governed jurisdiction


,candidate_jurisdiction,runner_rows,distinct_courses,first_date,last_date
0,Great Britain,3900,63,2025-10-15,2026-05-27
1,Ireland,1660,23,2025-10-15,2026-05-27
2,Hong Kong,199,2,2025-10-15,2026-03-08
3,France,78,7,2025-10-15,2026-03-19
4,Australia,40,8,2025-10-18,2025-12-06
5,Saudi Arabia,25,1,2026-02-14,2026-02-14
6,United States,14,5,2025-10-25,2025-12-28
7,Bahrain,5,1,2025-11-14,2026-02-19
8,New Zealand,5,3,2025-11-08,2026-01-03
9,Switzerland,3,1,2026-02-22,2026-02-22


## Stage 18 — Define the safe headgear interpretation boundary

The complete `hg` investigation supports the following governed interpretation.

### Blank values

A blank `hg` value means that no headgear code was supplied in this field.

It must not automatically be converted into a universal claim that the horse wore no equipment. The field may omit equipment outside the source coding scheme or reflect incomplete historical reporting.

### Equipment components

All 60 populated raw values can be decomposed into an ordered combination of:

- blinkers;
- cheekpieces;
- tongue-tie;
- hood;
- visor;
- eye hood;
- eyeshield;
- eyecover.

The raw source value and component order must remain preserved.

The source-specific component `c` can be normalised to `eyecover`, supported by the nine affected rows and the explicit Humble Spark comment. This does not imply that standalone `c` is a universal published racecard abbreviation.

### Trailing `1`

A trailing `1` is a source-presented first-time equipment declaration.

However:

- it appears only from 15 October 2025;
- it is not available consistently across the full historical period;
- no trailing `2` occurs in this source;
- database history cannot independently validate every declaration;
- the suffix may apply to one component within a combination.

Therefore a downstream model may preserve:

- the raw trailing suffix;
- a boolean source-declared-first-time flag;
- the equipment component combination.

It must not derive first-time status for earlier rows from absence of the suffix, nor claim that the source contains a complete lifetime equipment history.

In [19]:
# Build the final governed interpretation table for every observed raw hg
# value, including the blank source value.
#
# This table records what can be normalised safely while preserving the
# limitations established by the investigation.

headgear_governance_rows = []

# Represent the blank source value explicitly rather than silently treating it
# as equivalent to a fully verified "no equipment" declaration.
headgear_governance_rows.append(
    {
        "raw_hg": "",
        "runner_rows": 1_122_490,
        "normalised_components": "",
        "component_count": 0,
        "source_declared_first_time": False,
        "suffix_interpretation_status": "not_applicable",
        "interpretation_status": "blank_field_not_supplied",
        "normalisation_action": "preserve_blank",
        "evidence_basis": "source_profile",
    }
)

for row in governed_headgear_vocabulary.itertuples(index=False):
    has_trailing_1 = row.use_suffix == "1"

    headgear_governance_rows.append(
        {
            "raw_hg": row.raw_hg,
            "runner_rows": int(row.runner_rows),
            "normalised_components": row.normalised_components,
            "component_count": int(row.component_count),
            "source_declared_first_time": has_trailing_1,
            "suffix_interpretation_status": (
                "source_declared_first_time_from_2025_10_15"
                if has_trailing_1
                else "no_source_first_time_suffix"
            ),
            "interpretation_status": "fully_decomposed_source_code",
            "normalisation_action": (
                "preserve_raw_and_expose_components"
            ),
            "evidence_basis": (
                "NB17-HG-0001_and_source_context"
                if row.contains_source_eyecover_code
                else "NB17-HG-0001"
            ),
        }
    )

headgear_governance = pd.DataFrame(
    headgear_governance_rows
)

# Reconcile the complete blank and populated populations.
assert len(headgear_governance) == 61
assert headgear_governance["raw_hg"].is_unique
assert (
    headgear_governance["runner_rows"].sum()
    == EXPECTED_RUNNER_ROWS
)
assert (
    headgear_governance.loc[
        headgear_governance["raw_hg"].ne(""),
        "interpretation_status",
    ]
    == "fully_decomposed_source_code"
).all()

print("Final governed headgear vocabulary")
display(
    headgear_governance[
        [
            "raw_hg",
            "runner_rows",
            "normalised_components",
            "component_count",
            "source_declared_first_time",
            "suffix_interpretation_status",
            "interpretation_status",
            "normalisation_action",
            "evidence_basis",
        ]
    ]
)

# Summarise the complete runner population by governed interpretation.
headgear_governance_summary = (
    headgear_governance
    .groupby(
        [
            "interpretation_status",
            "normalisation_action",
        ],
        as_index=False,
    )
    .agg(
        distinct_raw_values=("raw_hg", "nunique"),
        runner_rows=("runner_rows", "sum"),
    )
)

headgear_governance_summary["runner_row_percentage"] = (
    headgear_governance_summary["runner_rows"]
    / EXPECTED_RUNNER_ROWS
    * 100
)

assert (
    headgear_governance_summary["runner_rows"].sum()
    == EXPECTED_RUNNER_ROWS
)

print("Headgear governance coverage")
display(headgear_governance_summary)

# Summarise how much of the complete runner population carries a source-
# declared first-time suffix.
headgear_first_time_summary = (
    headgear_governance
    .groupby(
        "source_declared_first_time",
        as_index=False,
    )
    .agg(
        distinct_raw_values=("raw_hg", "nunique"),
        runner_rows=("runner_rows", "sum"),
    )
)

headgear_first_time_summary["runner_row_percentage"] = (
    headgear_first_time_summary["runner_rows"]
    / EXPECTED_RUNNER_ROWS
    * 100
)

print("Source-declared first-time headgear coverage")
display(headgear_first_time_summary)

Final governed headgear vocabulary


,raw_hg,runner_rows,normalised_components,component_count,source_declared_first_time,suffix_interpretation_status,interpretation_status,normalisation_action,evidence_basis
0,,1122490,,0,False,not_applicable,blank_field_not_supplied,preserve_blank,source_profile
1,t,178727,tongue_tie,1,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
2,p,173713,cheekpieces,1,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
3,b,128859,blinkers,1,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
4,h,62258,hood,1,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
...,...,...,...,...,...,...,...,...,...
56,ev,1,eye_hood|visor,2,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
57,heb1,1,hood|eye_hood|blinkers,3,True,source_declared_first_time_from_2025_10_15,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
58,hetp,1,hood|eye_hood|tongue_tie|cheekpieces,4,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001
59,htbp,1,hood|tongue_tie|blinkers|cheekpieces,4,False,no_source_first_time_suffix,fully_decomposed_source_code,preserve_raw_and_expose_components,NB17-HG-0001


Headgear governance coverage


,interpretation_status,normalisation_action,distinct_raw_values,runner_rows,runner_row_percentage
0,blank_field_not_supplied,preserve_blank,1,1122490,60.63302
1,fully_decomposed_source_code,preserve_raw_and_expose_components,60,728795,39.36698


Source-declared first-time headgear coverage


,source_declared_first_time,distinct_raw_values,runner_rows,runner_row_percentage
0,False,39,1845353,99.679574
1,True,22,5932,0.320426


## Stage 19 — Preserve the source-specific eyecover evidence

The published reference defines eyecover as `e/c`, but the source contains no populated literal `e/c` value.

Instead, nine runner rows contain `c` within `hc`, `hct` and `cvp`. Their source context supports interpreting `c` as the source-specific eyecover component, most directly through Humble Spark’s comment that an eyeshield was worn instead of the declared eye cover.

This evidence authorises a downstream normalisation from source component `c` to `eyecover`.

It does not alter the immutable raw values and does not claim that standalone `c` is a universal racecard abbreviation.

In [20]:
# Persist the source-specific c-to-eyecover interpretation.
#
# This cell is restart-safe:
# - it appends the verification record when absent;
# - it accepts the already-persisted row when present;
# - it refuses to overwrite a permanent ID containing different evidence.

manual_verification_columns = [
    "verification_id",
    "subject_type",
    "source_date",
    "source_course",
    "source_off",
    "source_horse",
    "source_field",
    "raw_source_value",
    "verification_question",
    "verified_value",
    "verification_status",
    "evidence_type",
    "evidence_locator",
    "evidence_accessed_date",
    "governing_notebook",
    "confidence",
    "notes",
    "database_action",
]

MANUAL_VERIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "manual_verifications.csv"
)

eyecover_verification = pd.DataFrame(
    [
        {
            "verification_id": "NB17-HG-0002",
            "subject_type": "source_value",
            "source_date": "2023-05-30",
            "source_course": "Redcar",
            "source_off": "4:20",
            "source_horse": "Humble Spark (IRE)",
            "source_field": "hg",
            "raw_source_value": "c",
            "verification_question": (
                "What does the source-specific headgear component c mean "
                "within hc, hct and cvp?"
            ),
            "verified_value": "c=eyecover",
            "verification_status": "confirmed",
            "evidence_type": "source_context",
            "evidence_locator": (
                "source rowid 1347987; Humble Spark comment states "
                "'Wore eye shield instead of declared eye cover'"
            ),
            "evidence_accessed_date": "2026-07-31",
            "governing_notebook": "17",
            "confidence": "high",
            "notes": (
                "The source contains nine rows using c within hc, hct and "
                "cvp and no populated literal e/c values. Interpreting c as "
                "eyecover makes all three combinations coherent. This is a "
                "source-specific normalisation and does not establish c as "
                "a universal published abbreviation."
            ),
            "database_action": "reference_enrichment",
        }
    ],
    columns=manual_verification_columns,
)

manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    dtype=str,
    keep_default_na=False,
)

if list(manual_verifications.columns) != manual_verification_columns:
    raise ValueError(
        "Manual-verification register columns do not match "
        "the expected schema."
    )

verification_id = "NB17-HG-0002"

existing_eyecover_verification = (
    manual_verifications.loc[
        manual_verifications["verification_id"].eq(
            verification_id
        ),
        manual_verification_columns,
    ]
    .reset_index(drop=True)
)

if existing_eyecover_verification.empty:
    updated_manual_verifications = pd.concat(
        [
            manual_verifications,
            eyecover_verification,
        ],
        ignore_index=True,
    )

elif existing_eyecover_verification.equals(
    eyecover_verification
):
    updated_manual_verifications = manual_verifications.copy()

else:
    display(
        existing_eyecover_verification.compare(
            eyecover_verification,
            keep_shape=True,
            keep_equal=False,
        )
    )

    raise ValueError(
        "Existing NB17-HG-0002 differs from the "
        "verification record defined by this notebook."
    )

assert updated_manual_verifications[
    "verification_id"
].is_unique

updated_manual_verifications.to_csv(
    MANUAL_VERIFICATIONS_PATH,
    index=False,
)

# Reload the permanent register and confirm the persisted record.
reloaded_manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    dtype=str,
    keep_default_na=False,
)

reloaded_eyecover_verification = (
    reloaded_manual_verifications.loc[
        reloaded_manual_verifications[
            "verification_id"
        ].eq(verification_id)
    ]
    .reset_index(drop=True)
)

assert len(reloaded_eyecover_verification) == 1
assert (
    reloaded_eyecover_verification.loc[
        0,
        "verified_value",
    ]
    == "c=eyecover"
)
assert (
    reloaded_eyecover_verification.loc[
        0,
        "verification_status",
    ]
    == "confirmed"
)
assert (
    reloaded_eyecover_verification.loc[
        0,
        "database_action",
    ]
    == "reference_enrichment"
)

print(
    "Notebook 17 source-specific eyecover evidence "
    "persisted and reloaded"
)

display(
    reloaded_eyecover_verification[
        [
            "verification_id",
            "source_date",
            "source_course",
            "source_horse",
            "source_field",
            "raw_source_value",
            "verified_value",
            "verification_status",
            "confidence",
            "database_action",
        ]
    ]
)

Notebook 17 source-specific eyecover evidence persisted and reloaded


,verification_id,source_date,source_course,source_horse,source_field,raw_source_value,verified_value,verification_status,confidence,database_action
0,NB17-HG-0002,2023-05-30,Redcar,Humble Spark (IRE),hg,c,c=eyecover,confirmed,high,reference_enrichment


## Stage 20 — Field-level analytical decisions

### `age`

- Stored as an integer on all 1,851,285 governed runner rows.
- Nineteen distinct values are observed, ranging from `1` to `31`.
- The field is usable as the source-recorded runner age.
- Apparent conflicts with race-level `age_band` inherit Notebook 16’s limitations and must not be used here to overwrite runner age automatically.
- Extreme or contradictory values require targeted verification rather than global range clipping.

### `sex`

- Stored as text and populated on every governed runner row.
- Six common codes are verified and safely normalised:

  - `C` — colt;
  - `F` — filly;
  - `G` — gelding;
  - `H` — horse;
  - `M` — mare;
  - `R` — rig.

- Two isolated values were verified as source-field contamination:

  - `BB` for Par Coeur (GER) should reconcile to `G`;
  - `B` for La Venezolana (VEN) should reconcile to `F`.

- The immutable raw values must remain preserved, with corrections applied only through the governed verification layer.

### `hg`

- Blank on 1,122,490 rows and populated on 728,795 rows.
- All 60 populated raw values can be decomposed into ordered equipment components.
- Safe component normalisations are:

  - `b` — blinkers;
  - `p` — cheekpieces;
  - `t` — tongue-tie;
  - `h` — hood;
  - `v` — visor;
  - `e` — eye hood;
  - `e/s` — eyeshield;
  - source-specific `c` — eyecover.

- Raw combinations and component order must remain preserved.
- A trailing `1` may be retained as a source-declared first-time flag from 15 October 2025 onward.
- Absence of a suffix, especially before that date, must not be interpreted as a negative first-time declaration.
- Blank `hg` means no value was supplied in this field, not proven absence of all equipment.

In [22]:
# Consolidate the Notebook 17 field-level decisions into one auditable table.
#
# This table records the permitted downstream interpretation and the main
# limitations without altering the immutable source.

runner_characteristics_decisions = pd.DataFrame(
    [
        {
            "source_field": "age",
            "source_population": EXPECTED_RUNNER_ROWS,
            "distinct_raw_values": 19,
            "safe_interpretation": "source_recorded_runner_age",
            "normalisation_action": "preserve_integer_value",
            "correction_layer_required": False,
            "verification_ids": "",
            "main_limitation": (
                "Do not overwrite from age_band conflicts or clip extreme "
                "values without targeted verification."
            ),
        },
        {
            "source_field": "sex",
            "source_population": EXPECTED_RUNNER_ROWS,
            "distinct_raw_values": 8,
            "safe_interpretation": (
                "normalise C, F, G, H, M and R; reconcile two verified "
                "source-field contamination cases"
            ),
            "normalisation_action": (
                "preserve_raw_and_expose_verified_normalised_value"
            ),
            "correction_layer_required": True,
            "verification_ids": (
                "NB17-SEX-0001|NB17-SEX-0002|NB17-SEX-0003"
            ),
            "main_limitation": (
                "Raw B and BB must not be treated as additional sex "
                "categories; corrections require exact row lineage."
            ),
        },
        {
            "source_field": "hg",
            "source_population": EXPECTED_RUNNER_ROWS,
            "distinct_raw_values": 61,
            "safe_interpretation": (
                "preserve blank or decompose populated raw value into "
                "ordered governed equipment components"
            ),
            "normalisation_action": (
                "preserve_raw_and_expose_components_and_source_suffix"
            ),
            "correction_layer_required": False,
            "verification_ids": "NB17-HG-0001|NB17-HG-0002",
            "main_limitation": (
                "Blank means not supplied. Trailing 1 is available only "
                "from 2025-10-15 and must not be reconstructed historically."
            ),
        },
    ]
)

# Confirm the bounded Notebook 17 field set is represented exactly once.
assert set(
    runner_characteristics_decisions["source_field"]
) == {"age", "sex", "hg"}

assert runner_characteristics_decisions[
    "source_field"
].is_unique

assert (
    runner_characteristics_decisions["source_population"]
    == EXPECTED_RUNNER_ROWS
).all()

print("Notebook 17 field-level analytical decisions")
display(runner_characteristics_decisions)

Notebook 17 field-level analytical decisions


,source_field,source_population,distinct_raw_values,safe_interpretation,normalisation_action,correction_layer_required,verification_ids,main_limitation
0,age,1851285,19,source_recorded_runner_age,preserve_integer_value,False,,Do not overwrite from age_band conflicts or cl...
1,sex,1851285,8,"normalise C, F, G, H, M and R; reconcile two v...",preserve_raw_and_expose_verified_normalised_value,True,NB17-SEX-0001|NB17-SEX-0002|NB17-SEX-0003,Raw B and BB must not be treated as additional...
2,hg,1851285,61,preserve blank or decompose populated raw valu...,preserve_raw_and_expose_components_and_source_...,False,NB17-HG-0001|NB17-HG-0002,Blank means not supplied. Trailing 1 is availa...


## Stage 21 — Persist and reload the governed Notebook 17 outputs

The analytical decisions must not exist only in notebook memory.

This stage persists three compact, reusable reference outputs:

- the governed runner-sex vocabulary;
- the governed headgear vocabulary;
- the field-level analytical decision table.

The exports preserve raw source values alongside their authorised interpretations and limitations.

After writing, each file will be reloaded and reconciled to the in-memory result. This confirms that the persisted artifacts are complete and independently readable before reusable implementation is added.

In [23]:
# Persist the compact governed outputs produced by Notebook 17.
#
# A notebook-specific processed directory keeps these analytical artifacts
# separate from the immutable source and permanent manual-verification register.
NOTEBOOK_17_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_17_runner_characteristics"
)

NOTEBOOK_17_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SEX_VOCABULARY_OUTPUT_PATH = (
    NOTEBOOK_17_OUTPUT_DIR
    / "runner_sex_governance.csv"
)

HEADGEAR_VOCABULARY_OUTPUT_PATH = (
    NOTEBOOK_17_OUTPUT_DIR
    / "runner_headgear_governance.csv"
)

FIELD_DECISIONS_OUTPUT_PATH = (
    NOTEBOOK_17_OUTPUT_DIR
    / "runner_characteristics_decisions.csv"
)

# Build the final governed sex vocabulary.
#
# The two contaminated raw values remain explicit and carry exact permanent
# verification identifiers for downstream reconciliation.
sex_correction_map = {
    "B": {
        "normalised_sex": "filly",
        "interpretation_status": "verified_source_correction",
        "verification_id": "NB17-SEX-0003",
    },
    "BB": {
        "normalised_sex": "gelding",
        "interpretation_status": "verified_source_correction",
        "verification_id": "NB17-SEX-0002",
    },
}

runner_sex_governance = (
    sex_vocabulary_governance.copy()
)

for raw_sex, correction in sex_correction_map.items():
    correction_mask = (
        runner_sex_governance["raw_sex"].eq(raw_sex)
    )

    runner_sex_governance.loc[
        correction_mask,
        "normalised_sex",
    ] = correction["normalised_sex"]

    runner_sex_governance.loc[
        correction_mask,
        "interpretation_status",
    ] = correction["interpretation_status"]

    runner_sex_governance.loc[
        correction_mask,
        "verification_id",
    ] = correction["verification_id"]

runner_sex_governance["normalisation_action"] = (
    runner_sex_governance["interpretation_status"].map(
        {
            "verified_common_code": (
                "preserve_raw_and_expose_normalised_value"
            ),
            "verified_source_correction": (
                "preserve_raw_and_apply_verified_correction"
            ),
        }
    )
)

assert len(runner_sex_governance) == 8
assert runner_sex_governance["raw_sex"].is_unique
assert runner_sex_governance["normalised_sex"].notna().all()
assert (
    runner_sex_governance["runner_rows"].sum()
    == EXPECTED_RUNNER_ROWS
)

# Persist the three outputs.
runner_sex_governance.to_csv(
    SEX_VOCABULARY_OUTPUT_PATH,
    index=False,
)

headgear_governance.to_csv(
    HEADGEAR_VOCABULARY_OUTPUT_PATH,
    index=False,
)

runner_characteristics_decisions.to_csv(
    FIELD_DECISIONS_OUTPUT_PATH,
    index=False,
)

# Reload every artifact from disk without relying on notebook memory.
reloaded_runner_sex_governance = pd.read_csv(
    SEX_VOCABULARY_OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
)

reloaded_headgear_governance = pd.read_csv(
    HEADGEAR_VOCABULARY_OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
)

reloaded_runner_characteristics_decisions = pd.read_csv(
    FIELD_DECISIONS_OUTPUT_PATH,
    dtype=str,
    keep_default_na=False,
)

# Reconcile row counts and complete source populations after reload.
assert len(reloaded_runner_sex_governance) == 8
assert int(
    pd.to_numeric(
        reloaded_runner_sex_governance["runner_rows"]
    ).sum()
) == EXPECTED_RUNNER_ROWS

assert len(reloaded_headgear_governance) == 61
assert int(
    pd.to_numeric(
        reloaded_headgear_governance["runner_rows"]
    ).sum()
) == EXPECTED_RUNNER_ROWS

assert len(
    reloaded_runner_characteristics_decisions
) == 3

assert set(
    reloaded_runner_characteristics_decisions[
        "source_field"
    ]
) == {"age", "sex", "hg"}

print("Notebook 17 governed outputs persisted and reloaded")

display(
    pd.DataFrame(
        {
            "artifact": [
                "runner-sex governance",
                "runner-headgear governance",
                "field-level decisions",
            ],
            "path": [
                str(
                    SEX_VOCABULARY_OUTPUT_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
                str(
                    HEADGEAR_VOCABULARY_OUTPUT_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
                str(
                    FIELD_DECISIONS_OUTPUT_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
            ],
            "reloaded_rows": [
                len(reloaded_runner_sex_governance),
                len(reloaded_headgear_governance),
                len(
                    reloaded_runner_characteristics_decisions
                ),
            ],
        }
    )
)

Notebook 17 governed outputs persisted and reloaded


,artifact,path,reloaded_rows
0,runner-sex governance,data/processed/notebook_17_runner_characterist...,8
1,runner-headgear governance,data/processed/notebook_17_runner_characterist...,61
2,field-level decisions,data/processed/notebook_17_runner_characterist...,3


## Analytical conclusion and limitations

### Bounded question

What do the runner-level `age`, `sex` and `hg` fields represent in the source, how consistently are they populated, and which values can be normalised or derived safely without inventing runner identity, eligibility or equipment facts?

### Executive conclusion

The three fields are useful, but they require different treatment.

- `age` is a fully populated runner-level integer and can be retained as the source-recorded runner age. It must not be overwritten automatically from race-level `age_band` values or clipped merely because a value looks unusual.
- `sex` is fully populated and almost entirely governed by six standard codes. Two isolated rows contain colour information in the sex field and require exact, verification-backed corrections.
- `hg` is blank on most rows but every populated raw value can be decomposed into an ordered equipment combination. The field is suitable for component-level analysis provided that blanks, source-specific notation and the late introduction of first-time suffixes remain explicit.

### `age`

All 1,851,285 governed runner rows contain an integer `age` value.

Nineteen distinct values are observed, ranging from `1` to `31`. The field should be preserved as the source-recorded runner age rather than treated as independently verified biological or official eligibility evidence.

Notebook 16 already investigated race-level age conditions. Apparent conflicts between `age` and `age_band` do not justify automatic runner-age correction because they may reflect:

- incomplete or misleading race-level shorthand;
- jurisdiction-specific age conventions;
- extraction defects;
- isolated source errors.

Extreme values require targeted verification if they become material to a later analysis.

### `sex`

All governed runner rows contain a text `sex` value.

Six standard codes cover 1,851,283 rows, or 99.999892% of the population:

- `C` — colt;
- `F` — filly;
- `G` — gelding;
- `H` — horse;
- `M` — mare;
- `R` — rig.

Two isolated source values were externally contradicted:

- Par Coeur (GER), raw `BB`, verified as a gelding under `NB17-SEX-0002`;
- La Venezolana (VEN), raw `B`, verified as a filly at the affected race date under `NB17-SEX-0003`.

The raw values must remain immutable. Corrected values may be exposed only through the governed verification layer with exact source-row lineage.

`B` and `BB` must not be treated as additional general runner-sex categories.

### `hg`

The `hg` field is:

- blank on 1,122,490 rows, or 60.633020%;
- populated on 728,795 rows, or 39.366980%;
- represented by 60 distinct populated raw values.

Every populated value can be decomposed into an ordered combination of:

- blinkers;
- cheekpieces;
- tongue-tie;
- hood;
- visor;
- eye hood;
- eyeshield;
- eyecover.

The source-specific component `c` can be normalised to `eyecover`, supported by `NB17-HG-0002`. This affects nine rows across `hc`, `hct` and `cvp`. The raw component and its source-specific provenance must remain preserved.

Blank `hg` means no value was supplied in this field. It does not prove the absence of every possible item of equipment.

### Trailing `1`

A trailing `1` occurs on 5,932 runner rows across 22 raw values.

It first appears on 15 October 2025 and does not occur earlier in the source. No trailing `2` is observed.

The suffix may therefore be retained as a source-declared first-time flag from that date onward. Its absence must not be interpreted as a negative declaration, particularly before 15 October 2025.

Source history cannot independently prove lifetime first use because:

- historical reporting is incomplete;
- the convention was introduced late;
- the suffix may apply to one component within a combination;
- three suffixed rows have an exact unsuffixed value recorded earlier.

### Evidence classification

The conclusions combine:

- source facts: storage types, row counts, vocabularies and dates;
- derived interpretation: component decomposition and temporal coverage;
- external validation: standard codes and the bounded sex and eyecover decisions;
- inference: the trailing `1` behaves as a source declaration but is not a complete lifetime-history indicator.

Manual-verification decision: `captured`.

Permanent verification identifiers:

- `NB17-SEX-0001`;
- `NB17-SEX-0002`;
- `NB17-SEX-0003`;
- `NB17-HG-0001`;
- `NB17-HG-0002`.

### Analytical and betting limitations

The notebook justifies:

- retaining runner age as supplied;
- normalising the six standard sex codes;
- applying two exact verification-backed sex corrections;
- decomposing populated headgear codes into governed components;
- retaining the post-15 October 2025 source-declared first-time flag.

It does not justify:

- treating unusual ages as automatically wrong;
- deriving official race eligibility from runner age;
- interpreting blank `hg` as proven absence of equipment;
- deriving historical first-time equipment use from unsuffixed values;
- treating this source as a complete official equipment history;
- using the two contaminated sex values as general categories.

The immutable source remains unchanged.